# Online Retail — Customer & Product RFM Analysis

This dataset contains the invoices of a UK-based online store between **01/12/2010** and **09/12/2011**. According to the UCI description, many of its customers are wholesalers.

**Goals of this notebook**

1. Clean the data with justified decisions — nothing is dropped unless there is a clear reason.
2. Understand sales behaviour through interactive EDA with Plotly.
3. Run an **RFM analysis for customers** and split them into **4 main segments — Champions, Loyal, At Risk and Lost** — each one built from finer sub-segments (11 in total).
4. Run an **RFM analysis for products** to find the Stars and the Dead Stock.
5. Run **Apriori** to find which products are bought together.

**Notebook map**

| Section | Content |
|---|---|
| 1 | Setup & Loading |
| 2 | Data Audit |
| 3 | Separating real products from other lines (shipping, fees, adjustments...) |
| 4 | Standardising product descriptions |
| 5 | Recovering zero unit prices |
| 6 | CustomerID, duplicates, returns and date features |
| 7 | Interactive EDA |
| 8 | **Customer RFM & Segmentation** (4 main segments + 11 sub-segments, interactive filters) |
| 9 | Product RFM & Segmentation |
| 10 | **Market Basket Analysis (Apriori)** — products bought together |
| 11 | Summary & Limitations |

## 1. Setup & Loading

The original file is an Excel workbook, so I converted it to CSV once (the two commented lines in the cell below) to make reading much faster every time the notebook is run. From here on I work on `online_retail_raw.csv`.

In [1]:
# %pip install openpyxl

In [2]:
import pandas as pd
import numpy as np

In [3]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# One consistent look for every chart in the notebook
pio.templates.default = "plotly_white"

In [4]:
# df=pd.read_excel("Online Retail.xlsx")
# df.to_csv("online_retail_raw.csv", index=False)

In [5]:
df=pd.read_csv("online_retail_raw.csv")

## 2. Data Audit

Before any cleaning I need to know what I am looking at. Three things are done here: the first 20 rows, `info()`, and an audit table with each column's type, number of missing values and number of unique values.

What stood out:
- The data has **541,909 rows × 8 columns**, from 01/12/2010 to 09/12/2011.
- `CustomerID` is missing in **135,080 rows (24.93%)**, so about a quarter of the rows have no known customer.
- `Description` is missing in only 1,454 rows (0.27%).
- `InvoiceNo` is a string, not a number, because cancelled invoices start with the letter **C**. For the same reason `CustomerID` was converted to `Int64` (which accepts nulls) instead of float.

In [6]:
df.head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


In [8]:
df["CustomerID"] = pd.to_numeric(df["CustomerID"], errors="coerce").astype("Int64")

audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_%": df.isna().mean() * 100,
    "unique_values": df.nunique(),
})
display(audit)
print(f"Period: {df.InvoiceDate.min()}  →  {df.InvoiceDate.max()}")

,dtype,missing,missing_%,unique_values
InvoiceNo,str,0,0.000000,25900
StockCode,str,0,0.000000,4070
Description,str,1454,0.268311,4223
Quantity,int64,0,0.000000,722
InvoiceDate,str,0,0.000000,23260
UnitPrice,float64,0,0.000000,1630
CustomerID,Int64,135080,24.926694,4372
Country,str,0,0.000000,38


Period: 2010-12-01 08:26:00  →  2011-12-09 12:50:00


### 2.1 Revenue & Quality Checks

I created `LineRevenue = Quantity × UnitPrice` to measure the value of every line. Then a checks table counts the problems we expect in this data:

- **5,268** fully duplicated rows.
- **9,288** cancellation lines (the invoice starts with C), but **10,624** lines have a negative quantity. That means **1,336 lines have a negative quantity without being cancellations**, which need investigation.
- **2,515** lines with a price of 0, and two lines with a negative price.
- `StockCode` values that are not products, such as `POST`, `DOT`, `M`, `D`, `BANK CHARGES` and `AMAZONFEE`.

> These checks decide the cleaning steps in the following sections.

In [9]:
df["LineRevenue"] = df["Quantity"] * df["UnitPrice"]

In [10]:
is_cancel = df["InvoiceNo"].str.startswith("C")
is_nonproduct = ~df["StockCode"].str.match(r"^\d{5}")   # POST, D, M, BANK CHARGES...

checks = pd.Series({
    "Rows": len(df),
    "Exact duplicate rows": int(df.duplicated().sum()),
    "Invoices": df["InvoiceNo"].nunique(),
    "Customers (known)": df["CustomerID"].nunique(),
    "Cancellation lines (start with C)": int(is_cancel.sum()),
    "Negative Quantity lines": int((df["Quantity"] < 0).sum()),
    "Negative Quantity but NOT cancellation": int(((df["Quantity"] < 0) & ~is_cancel).sum()),
    "UnitPrice == 0": int((df["UnitPrice"] == 0).sum()),
    "UnitPrice < 0": int((df["UnitPrice"] < 0).sum()),
    "Lines without CustomerID": int(df["CustomerID"].isna().sum()),
    "Lines without Description": int(df["Description"].isna().sum()),
    "Non-product StockCodes": int(is_nonproduct.sum()),
})
display(checks.to_frame("count"))

print("\nNon-product StockCodes:")

nonprod = (df[is_nonproduct]
           .dropna(subset=["Description"])
           .groupby("StockCode")
           .agg(Description=("Description", lambda s: s.value_counts().index[0]),
                lines=("StockCode", "size"))
           .sort_values("lines", ascending=False))

display(nonprod.head(20))

,count
Rows,541909
Exact duplicate rows,5268
Invoices,25900
Customers (known),4372
Cancellation lines (start with C),9288
Negative Quantity lines,10624
Negative Quantity but NOT cancellation,1336
UnitPrice == 0,2515
UnitPrice < 0,2
Lines without CustomerID,135080



Non-product StockCodes:


,Description,lines
StockCode,,
POST,POSTAGE,1252
DOT,DOTCOM POSTAGE,709
M,Manual,571
C2,CARRIAGE,143
D,Discount,77
S,SAMPLES,63
BANK CHARGES,Bank Charges,37
AMAZONFEE,AMAZON FEE,34
CRUK,CRUK Commission,16


## 3. Separating Real Products from Non-Product Lines

RFM must be computed on **real products** only. Lines such as Shipping, Fees and Discounts do not describe buying behaviour, and they also distort Monetary (for example `AMAZONFEE` alone is about -221 thousand).

I built a dictionary that maps every unusual `StockCode` to its type (Shipping / Discount / Manual Adjustment / Samples / Fees ...); any other code is labelled **Product**. Then I compare the number of lines and the revenue of each type.

In [11]:
# Classify StockCodes based on their business nature

stockcode_categories = {
    "POST": "Shipping",
    "DOT": "Shipping",
    "C2": "Shipping",

    "D": "Discount",
    "M": "Manual Adjustment",
    "S": "Samples",

    "BANK CHARGES": "Financial Charges",
    "AMAZONFEE": "Fees",
    "CRUK": "Commission",

    "gift_0001_20": "Gift Voucher",
    "gift_0001_10": "Gift Voucher",
    "gift_0001_30": "Gift Voucher"
}

df["ItemType"] = (
    df["StockCode"]
    .map(stockcode_categories)
    .fillna("Product")
)

# Check classification
display(
    df.groupby("ItemType")
    .agg(
        Total_Lines=("InvoiceNo", "count"),
        Unique_StockCodes=("StockCode", "nunique"),
        Total_Revenue=("LineRevenue", "sum")
    )
    .sort_values("Total_Lines", ascending=False)
)

,Total_Lines,Unique_StockCodes,Total_Revenue
ItemType,,,
Product,538974,4058,9781917.633
Shipping,2110,3,279462.120
Manual Adjustment,571,1,-68674.190
Discount,77,1,-5696.220
Samples,63,1,-3049.390
Financial Charges,37,1,-7175.639
Fees,34,1,-221520.500
Gift Voucher,27,3,417.550
Commission,16,1,-7933.430


### 3.1 Negative Quantities & Prices

A negative quantity can be a return (cancellation) or something else. I separated the two:

- Of the negative quantities, **8,704** lines (86.69%) are cancellations and **1,336** lines (13.31%) are not.
- Only two lines have a negative UnitPrice, and they are not cancellations.

The strange part is the 1,336 lines, so I analyse them in the next cells.

In [12]:
# Isolate actual products
df_products = df[df["ItemType"] == "Product"].copy()

# Identify cancellation invoices
df_products["IsCancellation"] = (
    df_products["InvoiceNo"]
    .astype("string")
    .str.startswith("C", na=False)
)

# Define negative conditions
negative_conditions = {
    "Negative Quantity": df_products["Quantity"] < 0,
    "Negative UnitPrice": df_products["UnitPrice"] < 0
}

results = []

for metric, condition in negative_conditions.items():

    negative_data = df_products[condition]
    total = len(negative_data)

    for is_cancel, label in [
        (True, "Cancellation"),
        (False, "Non-Cancellation")
    ]:

        count = (
            negative_data["IsCancellation"] == is_cancel
        ).sum()

        results.append({
            "Metric": metric,
            "Transaction_Type": label,
            "Count": count,
            "Percentage": (
                count / total * 100 if total else 0
            )
        })

negative_comparison = pd.DataFrame(results)

display(
    negative_comparison.pivot(
        index="Metric",
        columns="Transaction_Type",
        values=["Count", "Percentage"]
    ).round(2)
)

Count                    Percentage                 
Transaction_Type   Cancellation Non-Cancellation Cancellation Non-Cancellation
Metric                                                                        
Negative Quantity        8704.0           1336.0        86.69            13.31
Negative UnitPrice          0.0              2.0         0.00           100.00

I inspect these 1,336 lines directly: the largest negative quantities and their descriptions (some are `damages`, `printing smudges/thrown away` and `check`). Then I cross-tabulate the price type against the CustomerID status, and finally I inspect **all the lines of these invoices**, not only the negative line.

In [13]:
# 1. Non-cancellation transactions with negative Quantity

non_cancelled_qty = df_products[
    (~df_products["IsCancellation"]) &
    (df_products["Quantity"] < 0)
].copy()

# Group by StockCode and Description
negative_products = (
    non_cancelled_qty
    .groupby(["StockCode", "Description"])
    .agg(
        Negative_Lines=("InvoiceNo", "count"),
        Unique_Invoices=("InvoiceNo", "nunique"),
        Total_Negative_Quantity=("Quantity", "sum")
    )
    .reset_index()
    .sort_values("Negative_Lines", ascending=False)
)

display(negative_products)

,StockCode,Description,Negative_Lines,Unique_Invoices,Total_Negative_Quantity
319,72802B,check,3,3,-290
320,72802C,check,3,3,-75
213,22943,check,2,2,-33
154,22423,damages,2,2,-40
221,23005,printing smudges/thrown away,2,2,-19200
...,...,...,...,...,...
11,20694,?,1,1,-110
12,20700,?,1,1,-121
13,20711,check,1,1,-220
14,20711,lost in space,1,1,-98


In [14]:
# Inspect transactions with the largest negative quantities

display(
    non_cancelled_qty
    .sort_values("Quantity")
    [[
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "UnitPrice",
        "CustomerID",
        "InvoiceDate"
    ]]
    .head(20)
)

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,InvoiceDate
225529,556690,23005,printing smudges/thrown away,-9600,0.0,<NA>,2011-06-14 10:37:00
225530,556691,23005,printing smudges/thrown away,-9600,0.0,<NA>,2011-06-14 10:37:00
225528,556687,23003,Printing smudges/thrown away,-9058,0.0,<NA>,2011-06-14 10:36:00
115818,546152,72140F,throw away,-5368,0.0,<NA>,2011-03-09 17:25:00
431381,573596,79323W,"Unsaleable, destroyed.",-4830,0.0,<NA>,2011-10-31 15:17:00
341601,566768,16045,NaN,-3667,0.0,<NA>,2011-09-14 17:53:00
323458,565304,16259,NaN,-3167,0.0,<NA>,2011-09-02 12:18:00
263884,560039,20713,wrongly marked. 23343 in box,-3100,0.0,<NA>,2011-07-14 14:27:00
113580,545990,84598,check,-3000,0.0,<NA>,2011-03-08 13:07:00
375429,569466,23270,incorrect stock entry.,-2880,0.0,<NA>,2011-10-04 11:42:00


In [15]:
# Analyze non-cancellation negative transactions

negative_audit = non_cancelled_qty.copy()

negative_audit["Price_Status"] = np.select(
    [
        negative_audit["UnitPrice"] < 0,
        negative_audit["UnitPrice"] == 0,
        negative_audit["UnitPrice"] > 0
    ],
    [
        "Negative Price",
        "Zero Price",
        "Positive Price"
    ],
    default="Unknown"
)

negative_audit["Customer_Status"] = np.where(
    negative_audit["CustomerID"].isna(),
    "Missing CustomerID",
    "Has CustomerID"
)

# Cross analysis
display(
    pd.crosstab(
        negative_audit["Price_Status"],
        negative_audit["Customer_Status"],
        margins=True
    )
)

Customer_Status,Missing CustomerID,All
Price_Status,,
Zero Price,1336,1336
All,1336,1336


In [16]:
# Inspect all transactions belonging to these invoices

negative_invoice_ids = non_cancelled_qty["InvoiceNo"].unique()

related_invoice_rows = df[
    df["InvoiceNo"].isin(negative_invoice_ids)
].copy()

invoice_check = (
    related_invoice_rows
    .groupby("InvoiceNo")
    .agg(
        Invoice_Lines=("StockCode", "count"),

        Negative_Lines=(
            "Quantity",
            lambda x: (x < 0).sum()
        ),

        Positive_Lines=(
            "Quantity",
            lambda x: (x > 0).sum()
        ),

        Has_CustomerID=(
            "CustomerID",
            lambda x: x.notna().any()
        ),

        All_Zero_Price=(
            "UnitPrice",
            lambda x: x.eq(0).all()
        )
    )
)

invoice_summary = pd.Series({
    "Total Related Invoices": len(invoice_check),

    "Invoices With Positive Quantities":
        (invoice_check["Positive_Lines"] > 0).sum(),

    "Invoices With CustomerID":
        invoice_check["Has_CustomerID"].sum(),

    "Invoices With Non-Zero Prices":
        (~invoice_check["All_Zero_Price"]).sum()
})

display(invoice_summary.to_frame("Count"))

display(invoice_check.head(10))

,Count
Total Related Invoices,1336
Invoices With Positive Quantities,0
Invoices With CustomerID,0
Invoices With Non-Zero Prices,0


,Invoice_Lines,Negative_Lines,Positive_Lines,Has_CustomerID,All_Zero_Price
InvoiceNo,,,,,
536589,1,1,0,False,True
536764,1,1,0,False,True
536996,1,1,0,False,True
536997,1,1,0,False,True
536998,1,1,0,False,True
537000,1,1,0,False,True
537001,1,1,0,False,True
537003,1,1,0,False,True
537004,1,1,0,False,True


In [17]:
non_cancelled_price = df_products[
    (~df_products["IsCancellation"]) &
    (df_products["UnitPrice"] < 0)
]

display(
    non_cancelled_price[
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice",
            "CustomerID",
            "InvoiceDate"
        ]
    ]
)


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,InvoiceDate
299983,A563186,B,Adjust bad debt,1,-11062.06,<NA>,2011-08-12 14:51:00
299984,A563187,B,Adjust bad debt,1,-11062.06,<NA>,2011-08-12 14:52:00


**Conclusion from the inspection:** all 1,336 lines have a price of zero and no `CustomerID`, and their invoices contain no positive line and no non-zero price. So these are neither sales nor customer returns; they are **inventory adjustments** (damaged / lost / stock check).  
The two lines with a negative price have `StockCode = B` and the description *Adjust bad debt*, which is a **financial adjustment**.

That is why I classified them into two new types, `Inventory Adjustment` and `Financial Adjustment`, which leaves **537,636** genuine Product lines.

In [18]:
# 1. Identify Inventory Adjustments

is_cancellation = (
    df["InvoiceNo"]
    .astype("string")
    .str.startswith("C", na=False)
)

is_inventory_adjustment = (
    (df["Quantity"] < 0) &
    (df["UnitPrice"] == 0) &
    (df["CustomerID"].isna()) &
    (~is_cancellation)
)

# Classify Inventory Adjustments
df.loc[
    is_inventory_adjustment,
    "ItemType"
] = "Inventory Adjustment"


# 2. Identify Financial Adjustments (Bad Debt)

is_bad_debt = (
    (df["StockCode"] == "B") &
    df["Description"].str.contains(
        "Adjust bad debt",
        case=False,
        na=False
    ) &
    (df["UnitPrice"] < 0)
)

# Classify Financial Adjustments
df.loc[
    is_bad_debt,
    "ItemType"
] = "Financial Adjustment"


# 3. Check Classification

display(
    df.groupby("ItemType")
    .agg(
        Total_Lines=("InvoiceNo", "count"),
        Unique_StockCodes=("StockCode", "nunique"),
        Total_Quantity=("Quantity", "sum")
    )
    .sort_values("Total_Lines", ascending=False)
)

print("\nInventory Adjustments:", is_inventory_adjustment.sum())
print("Financial Adjustments:", is_bad_debt.sum())

,Total_Lines,Unique_StockCodes,Total_Quantity
ItemType,,,
Product,537636,3945,5373107
Shipping,2110,3,8350
Inventory Adjustment,1336,1082,-206957
Manual Adjustment,571,1,3164
Discount,77,1,-1194
Samples,63,1,-59
Financial Charges,37,1,-13
Fees,34,1,-30
Gift Voucher,27,3,96



Inventory Adjustments: 1336
Financial Adjustments: 2


This chart shows the distribution of line types after the whole classification. I used a log scale because Product dominates by a huge margin and the other types would otherwise be invisible.

In [19]:
# Lines per ItemType after all the classification steps
itemtype_summary = (
    df.groupby("ItemType")
    .agg(Lines=("InvoiceNo", "size"), Revenue=("LineRevenue", "sum"))
    .sort_values("Lines")
    .reset_index()
)

fig = px.bar(
    itemtype_summary,
    x="Lines", y="ItemType", orientation="h",
    log_x=True, text="Lines",
    hover_data={"Revenue": ":,.0f"},
    title="Row Classification: Lines per ItemType (log scale)",
    labels={"Lines": "Number of lines (log scale)", "ItemType": ""},
)
fig.update_traces(texttemplate="%{text:,}", textposition="outside", marker_color="#2E86DE")
fig.update_layout(title_x=0.5, height=430, showlegend=False)
fig.show()

In [20]:
# Isolate actual products

df_products = df[
    df["ItemType"] == "Product"
].copy()

print("Total Product Transactions:", len(df_products))

Total Product Transactions: 537636


## 4. Cleaning Product Descriptions

Every `StockCode` should have exactly one description, but in reality:
- **339** codes have more than one Description.
- Some of them are staff notes rather than descriptions (`check`, `found`, `damages`...), and some are only spelling differences (a comma, an extra space).

Unifying the description matters because product names are displayed in the charts and in the RFM.

In [21]:
# Analyze StockCodes with multiple descriptions

description_analysis = (
    df_products.dropna(subset=["Description"])
    .groupby(["StockCode", "Description"])
    .size()
    .reset_index(name="Frequency")
)

multiple_description_codes = (
    description_analysis.groupby("StockCode")["Description"]
    .nunique()
)

multiple_description_codes = multiple_description_codes[
    multiple_description_codes > 1
].index

multiple_descriptions = description_analysis[
    description_analysis["StockCode"].isin(
        multiple_description_codes
    )
].sort_values(
    ["StockCode", "Frequency"],
    ascending=[True, False]
)

print("StockCodes with Multiple Descriptions:",
      len(multiple_description_codes))

sample_codes = multiple_description_codes[:20]

display(
    multiple_descriptions[
        multiple_descriptions["StockCode"].isin(sample_codes)
    ]
)

StockCodes with Multiple Descriptions: 339


,StockCode,Description,Frequency
1,10080,GROOVY CACTUS INFLATABLE,22
2,10080,check,1
47,16156L,WRAP CAROUSEL,14
48,16156L,"WRAP, CAROUSEL",4
55,16162M,THE KING GIFT BAG 25x24x12cm,8
56,16162M,alan hodge cant mamage this section,1
57,16168M,FUNKY MONKEY GIFT BAG MEDIUM,41
58,16168M,found,1
68,16207A,PINK STRAWBERRY HANDBAG,34
69,16207A,check,1


**Step 1 — Exclude descriptions that are not product descriptions:** I built a keyword list (`check`, `found`, `damaged`, `lost`, `missing`, ...) and found the descriptions that contain them. Then I listed the codes that still have more than one "valid" description: **262** codes remain.

In [22]:
# Common non-product descriptions

invalid_keywords = [
    "check",
    "found",
    "adjustment",
    "dotcom",
    "wrongly",
    "marked as",
    "marked",
    "damages",
    "damaged",
    "lost",
    "missing"
]

description_analysis["Description_Clean"] = (
    description_analysis["Description"].str.strip()
)

invalid_mask = description_analysis["Description_Clean"].str.contains(
    "|".join(invalid_keywords),
    case=False,
    na=False
)

display(
    description_analysis[invalid_mask]
)

,StockCode,Description,Frequency,Description_Clean
2,10080,check,1,check
58,16168M,found,1,found
69,16207A,check,1,check
94,17012F,check,1,check
108,17091A,found,1,found
...,...,...,...,...
3833,85135B,check,1,check
3835,85135C,found,1,found
3881,85185B,Found in w/hse,1,Found in w/hse
4319,gift_0001_40,Dotcomgiftshop Gift Voucher £40.00,3,Dotcomgiftshop Gift Voucher £40.00


In [23]:
valid_descriptions = description_analysis[
    ~description_analysis.index.isin(
        description_analysis[invalid_mask].index
    )
].copy()

mode_description_map = (
    valid_descriptions
    .sort_values("Frequency", ascending=False)
    .drop_duplicates("StockCode")
    .set_index("StockCode")["Description"]
)

In [24]:
# Identify StockCodes with multiple valid descriptions

remaining_multiple_codes = (
    valid_descriptions.groupby("StockCode")["Description"]
    .nunique()
)

remaining_multiple_codes = remaining_multiple_codes[
    remaining_multiple_codes > 1
].index

remaining_multiple_descriptions = valid_descriptions[
    valid_descriptions["StockCode"].isin(remaining_multiple_codes)
].sort_values(
    ["StockCode", "Frequency"],
    ascending=[True, False]
)

print("Remaining StockCodes with Multiple Descriptions:",
      len(remaining_multiple_codes))

display(remaining_multiple_descriptions)

Remaining StockCodes with Multiple Descriptions: 262


,StockCode,Description,Frequency,Description_Clean
47,16156L,WRAP CAROUSEL,14,WRAP CAROUSEL
48,16156L,"WRAP, CAROUSEL",4,"WRAP, CAROUSEL"
55,16162M,THE KING GIFT BAG 25x24x12cm,8,THE KING GIFT BAG 25x24x12cm
56,16162M,alan hodge cant mamage this section,1,alan hodge cant mamage this section
113,17107D,"FLOWER FAIRY,5 SUMMER B'DRAW LINERS",25,"FLOWER FAIRY,5 SUMMER B'DRAW LINERS"
...,...,...,...,...
3955,90014A,SILVER M.O.P. ORBIT NECKLACE,6,SILVER M.O.P. ORBIT NECKLACE
3957,90014B,GOLD M PEARL ORBIT NECKLACE,13,GOLD M PEARL ORBIT NECKLACE
3958,90014B,GOLD M.O.P. ORBIT NECKLACE,2,GOLD M.O.P. ORBIT NECKLACE
3959,90014C,SILVER AND BLACK ORBIT NECKLACE,2,SILVER AND BLACK ORBIT NECKLACE


**Step 2 — Normalization:** I converted the description to uppercase and removed punctuation and extra spaces to find out whether the difference is only spelling. Result: only **21** codes became a single description, and **241** are still genuinely different in wording. The 21 were unified to the most frequent description.

In [25]:
import re

def normalize_description(text):
    text = str(text).upper().strip()

    # Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text)

    return text


remaining_multiple_descriptions["Normalized_Description"] = (
    remaining_multiple_descriptions["Description"]
    .apply(normalize_description)
)

normalization_analysis = (
    remaining_multiple_descriptions
    .groupby("StockCode")
    .agg(
        Original_Descriptions=("Description", "nunique"),
        Normalized_Descriptions=("Normalized_Description", "nunique")
    )
)

normalization_analysis["Can_Unify"] = (
    normalization_analysis["Normalized_Descriptions"] == 1
)

print("StockCodes Before Normalization:",
      len(normalization_analysis))

print("StockCodes Unifiable:",
      normalization_analysis["Can_Unify"].sum())

print("StockCodes Still Different:",
      (~normalization_analysis["Can_Unify"]).sum())

display(normalization_analysis.head(20))

StockCodes Before Normalization: 262
StockCodes Unifiable: 21
StockCodes Still Different: 241


,Original_Descriptions,Normalized_Descriptions,Can_Unify
StockCode,,,
16156L,2,1,True
16162M,2,2,False
17107D,3,3,False
20622,2,2,False
20725,2,2,False
20914,2,2,False
21098,2,2,False
21109,2,1,True
21112,2,1,True


In [26]:
# Get StockCodes that have identical descriptions after normalization

unifiable_codes = normalization_analysis[
    normalization_analysis["Can_Unify"]
].index

unifiable_samples = (
    remaining_multiple_descriptions[
        remaining_multiple_descriptions["StockCode"].isin(unifiable_codes)
    ]
    .sort_values(["StockCode", "Frequency"], ascending=[True, False])
)

display(unifiable_samples)

,StockCode,Description,Frequency,Description_Clean,Normalized_Description
47,16156L,WRAP CAROUSEL,14,WRAP CAROUSEL,WRAP CAROUSEL
48,16156L,"WRAP, CAROUSEL",4,"WRAP, CAROUSEL",WRAP CAROUSEL
397,21109,LARGE CAKE TOWEL CHOCOLATE SPOTS,109,LARGE CAKE TOWEL CHOCOLATE SPOTS,LARGE CAKE TOWEL CHOCOLATE SPOTS
398,21109,"LARGE CAKE TOWEL, CHOCOLATE SPOTS",8,"LARGE CAKE TOWEL, CHOCOLATE SPOTS",LARGE CAKE TOWEL CHOCOLATE SPOTS
402,21112,"SWISS ROLL TOWEL, PINK SPOTS",43,"SWISS ROLL TOWEL, PINK SPOTS",SWISS ROLL TOWEL PINK SPOTS
401,21112,SWISS ROLL TOWEL PINK SPOTS,39,SWISS ROLL TOWEL PINK SPOTS,SWISS ROLL TOWEL PINK SPOTS
497,21243,PINK POLKADOT PLATE,187,PINK POLKADOT PLATE,PINK POLKADOT PLATE
498,21243,PINK POLKADOT PLATE,25,PINK POLKADOT PLATE,PINK POLKADOT PLATE
677,21507,"ELEPHANT, BIRTHDAY CARD,",168,"ELEPHANT, BIRTHDAY CARD,",ELEPHANT BIRTHDAY CARD
676,21507,ELEPHANT BIRTHDAY CARD,97,ELEPHANT BIRTHDAY CARD,ELEPHANT BIRTHDAY CARD


In [27]:
# Build the most frequent description for each unifiable StockCode

unifiable_description_map = (
    unifiable_samples
    .sort_values(
        ["StockCode", "Frequency"],
        ascending=[True, False]
    )
    .drop_duplicates("StockCode")
    .set_index("StockCode")["Description"]
)

# Keep original data and update only the 21 StockCodes

df_products["Description"] = df_products["Description"].mask(
    df_products["StockCode"].isin(unifiable_codes),
    df_products["StockCode"].map(unifiable_description_map)
)

print("Unified StockCodes:", len(unifiable_codes))

Unified StockCodes: 21


In [28]:
# Select StockCodes with the highest number of different descriptions

review_codes = (
    remaining_multiple_descriptions
    .groupby("StockCode")
    .agg(
        Description_Count=("Description", "nunique"),
        Total_Frequency=("Frequency", "sum")
    )
    .sort_values(
        ["Description_Count", "Total_Frequency"],
        ascending=False
    )
    .head(20)
    .index
)

# Display all descriptions for selected StockCodes

review_sample = (
    remaining_multiple_descriptions[
        remaining_multiple_descriptions["StockCode"].isin(review_codes)
    ]
    .sort_values(
        ["StockCode", "Frequency"],
        ascending=[True, False]
    )
)

# Display all remaining StockCodes with multiple descriptions

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(
    remaining_multiple_descriptions.head(40)
    .sort_values(
        ["StockCode", "Frequency"],
        ascending=[True, False]
    )
)

,StockCode,Description,Frequency,Description_Clean,Normalized_Description
47,16156L,WRAP CAROUSEL,14,WRAP CAROUSEL,WRAP CAROUSEL
48,16156L,"WRAP, CAROUSEL",4,"WRAP, CAROUSEL",WRAP CAROUSEL
55,16162M,THE KING GIFT BAG 25x24x12cm,8,THE KING GIFT BAG 25x24x12cm,THE KING GIFT BAG 25X24X12CM
56,16162M,alan hodge cant mamage this section,1,alan hodge cant mamage this section,ALAN HODGE CANT MAMAGE THIS SECTION
113,17107D,"FLOWER FAIRY,5 SUMMER B'DRAW LINERS",25,"FLOWER FAIRY,5 SUMMER B'DRAW LINERS",FLOWER FAIRY5 SUMMER BDRAW LINERS
111,17107D,FLOWER FAIRY 5 DRAWER LINERS,21,FLOWER FAIRY 5 DRAWER LINERS,FLOWER FAIRY 5 DRAWER LINERS
112,17107D,FLOWER FAIRY 5 SUMMER DRAW LINERS,1,FLOWER FAIRY 5 SUMMER DRAW LINERS,FLOWER FAIRY 5 SUMMER DRAW LINERS
135,20622,VIPPASSPORT COVER,35,VIPPASSPORT COVER,VIPPASSPORT COVER
134,20622,VIP PASSPORT COVER,17,VIP PASSPORT COVER,VIP PASSPORT COVER
193,20725,LUNCH BAG RED RETROSPOT,1638,LUNCH BAG RED RETROSPOT,LUNCH BAG RED RETROSPOT


**Step 3 — Most frequent description:** the remaining codes were unified to the most frequent description of each `StockCode`. The next cell verifies that no code is left with more than one description.

I also noticed that the number of descriptions (3,804) is smaller than the number of codes (3,945), which means different codes share the same description. That is why I treat **`StockCode` as the product ID** in every analysis, not the description.

In [29]:
# Select the most frequent description for each StockCode

description_mapping = (
    df_products
    .dropna(subset=["Description"])
    .groupby(["StockCode", "Description"])
    .size()
    .reset_index(name="Frequency")
    .sort_values(
        ["StockCode", "Frequency"],
        ascending=[True, False]
    )
    .drop_duplicates(subset="StockCode")
    .set_index("StockCode")["Description"]
)

# Replace descriptions with the most frequent description
df_products["Description"] = (
    df_products["StockCode"]
    .map(description_mapping)
)

print("Unique StockCodes:", df_products["StockCode"].nunique())
print("Unique Descriptions:", df_products["Description"].nunique())


Unique StockCodes: 3945
Unique Descriptions: 3804


In [30]:
# Validate description consistency per StockCode

description_check = (
    df_products
    .groupby("StockCode")["Description"]
    .nunique()
)

print("StockCodes with Multiple Descriptions:",
      (description_check > 1).sum())

print("StockCodes with One Description:",
      (description_check == 1).sum())

print("StockCodes with Missing Description:",
      df_products.groupby("StockCode")["Description"]
      .apply(lambda x: x.isna().all()).sum())

StockCodes with Multiple Descriptions: 0
StockCodes with One Description: 3929
StockCodes with Missing Description: 16


**Step 4 — Missing descriptions:** 16 codes have no description in any row, and there is no source to recover it from. I set them to `Unknown Product` and created a `Description_Missing` column to flag them, so that it stays clear what was modified.

In [31]:
# Recover Missing Descriptions using StockCode

# Clean descriptions before building the mapping
df_products["Description"] = (
    df_products["Description"]
    .str.strip()
)

# Get the most frequent non-missing Description for each StockCode
description_mapping = (
    df_products
    .dropna(subset=["Description"])
    .groupby(["StockCode", "Description"])
    .size()
    .reset_index(name="Frequency")
    .sort_values(
        ["StockCode", "Frequency"],
        ascending=[True, False]
    )
    .drop_duplicates("StockCode")
    .set_index("StockCode")["Description"]
)

# Identify missing descriptions
missing_mask = df_products["Description"].isna()

# Fill missing descriptions using StockCode
df_products.loc[missing_mask, "Description"] = (
    df_products.loc[missing_mask, "StockCode"]
    .map(description_mapping)
)

# Calculate results
recovered = missing_mask.sum() - df_products["Description"].isna().sum()
remaining = df_products["Description"].isna().sum()

print("Original Missing Descriptions:", missing_mask.sum())
print("Successfully Recovered:", recovered)
print("Remaining Missing Descriptions:", remaining)

# Display remaining missing descriptions
remaining_missing = df_products[
    df_products["Description"].isna()
].copy()

print("Remaining Missing Transactions:", len(remaining_missing))

display(
    remaining_missing[
        [
            "InvoiceNo",
            "StockCode",
            "Quantity",
            "UnitPrice",
            "CustomerID",
            "InvoiceDate"
        ]
    ]
)


Original Missing Descriptions: 16
Successfully Recovered: 0
Remaining Missing Descriptions: 16
Remaining Missing Transactions: 16


,InvoiceNo,StockCode,Quantity,UnitPrice,CustomerID,InvoiceDate
1970,536545,21134,1,0.0,<NA>,2010-12-01 14:32:00
1987,536549,85226A,1,0.0,<NA>,2010-12-01 14:34:00
1988,536550,85044,1,0.0,<NA>,2010-12-01 14:34:00
2024,536552,20950,1,0.0,<NA>,2010-12-01 14:34:00
2026,536554,84670,23,0.0,<NA>,2010-12-01 14:35:00
7187,536995,35951,57,0.0,<NA>,2010-12-03 15:29:00
19628,537875,20849,1,0.0,<NA>,2010-12-08 18:08:00
19631,537878,72803B,1,0.0,<NA>,2010-12-08 18:09:00
21782,538133,85018C,3,0.0,<NA>,2010-12-09 15:56:00
21786,538137,62095B,2,0.0,<NA>,2010-12-09 15:57:00


In [32]:
# Handle descriptions that could not be recovered

# Keep track of records with unrecoverable descriptions
df_products["Description_Missing"] = (
    df_products["Description"].isna()
)

# Replace remaining missing descriptions
df_products["Description"] = (
    df_products["Description"]
    .fillna("Unknown Product")
)

print("Remaining Missing Descriptions:",
      df_products["Description"].isna().sum())

print("Unknown Product Transactions:",
      (df_products["Description"] == "Unknown Product").sum())

print("Total Transactions:", len(df_products))

Remaining Missing Descriptions: 0
Unknown Product Transactions: 16
Total Transactions: 537636


## 5. Recovering Zero Unit Prices

There are **1,162** product lines with a price of 0 (only 0.22%). A price of 0 means zero revenue even though a quantity was sold, so I try to recover the price instead of throwing the row away.

> This is an **assumption**: that a price of 0 means a missing price, not a free gift. The prices I recover are therefore estimates, not real prices.

In [33]:
# Identify products with zero UnitPrice

zero_price_products = df_products[
    df_products["UnitPrice"] == 0
].copy()

# Summary

total_products = len(df_products)
zero_price_count = len(zero_price_products)

zero_price_summary = pd.DataFrame({
    "Count": [
        total_products,
        zero_price_count,
        total_products - zero_price_count
    ],
    "Percentage": [
        100,
        zero_price_count / total_products * 100,
        (total_products - zero_price_count) / total_products * 100
    ]
}, index=[
    "Total Product Transactions",
    "Zero UnitPrice",
    "Non-Zero UnitPrice"
])

display(zero_price_summary.round(2))

,Count,Percentage
Total Product Transactions,537636,100.00
Zero UnitPrice,1162,0.22
Non-Zero UnitPrice,536474,99.78


**Attempt 1 — Same description:** if the description has exactly one positive price in the whole dataset, I take that price. Only **135** lines were recovered, because most descriptions had their price changed more than once (**3,300** descriptions have more than one price).

In [34]:
# 1. Create a copy
df_products_clean = df_products.copy()

# 2. Normalize descriptions for matching
df_products_clean["_Description_Key"] = (
    df_products_clean["Description"]
    .str.strip()
    .str.upper()
)

# 3. Get products with known positive prices
valid_prices = df_products_clean[
    df_products_clean["UnitPrice"] > 0
].copy()

# 4. Analyze available prices for each description
description_prices = (
    valid_prices
    .groupby("_Description_Key")["UnitPrice"]
    .agg(
        Unique_Prices="nunique",
        Suggested_Price="median"
    )
)

# 5. Identify descriptions having exactly one known price
unique_price_map = description_prices[
    description_prices["Unique_Prices"] == 1
]["Suggested_Price"]

# 6. Identify zero-price transactions
zero_mask = df_products_clean["UnitPrice"] == 0

# 7. Match zero prices using Description
suggested_prices = (
    df_products_clean.loc[zero_mask, "_Description_Key"]
    .map(unique_price_map)
)

# 8. Fill only unambiguous matches
fillable = suggested_prices.notna()

df_products_clean.loc[
    suggested_prices[fillable].index,
    "UnitPrice"
] = suggested_prices[fillable]

# 9. Results
print("Original Zero Prices:", zero_mask.sum())
print("Successfully Recovered:", fillable.sum())
print("Remaining Zero Prices:",
      (df_products_clean["UnitPrice"] == 0).sum())

print(
    "Descriptions With Multiple Prices:",
    (description_prices["Unique_Prices"] > 1).sum()
)

# 10. Remove helper column
df_products_clean.drop(
    columns="_Description_Key",
    inplace=True
)

Original Zero Prices: 1162
Successfully Recovered: 135
Remaining Zero Prices: 1027
Descriptions With Multiple Prices: 3300


C:\Users\hamas\AppData\Local\Temp\ipykernel_22640\2984657450.py:60: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_products_clean.drop(


**Attempt 2 — Nearest price in time:** for the same `StockCode`, I look for the last positive price before the date and the first one after it (`merge_asof`) and take the one that is closest in time. This recovered **1,008** lines.

In [35]:
# 1. Get remaining zero-price transactions
remaining_zero = df_products_clean[
    df_products_clean["UnitPrice"] == 0
].copy()

remaining_zero["_target_index"] = remaining_zero.index
remaining_zero = remaining_zero.reset_index(drop=True)

# 2. Get positive prices
valid_prices = df_products[
    df_products["UnitPrice"] > 0
][["StockCode", "InvoiceDate", "UnitPrice"]].copy()

# 3. Standardize data types
remaining_zero["InvoiceDate"] = pd.to_datetime(
    remaining_zero["InvoiceDate"], errors="coerce"
)

valid_prices["InvoiceDate"] = pd.to_datetime(
    valid_prices["InvoiceDate"], errors="coerce"
)

remaining_zero["StockCode"] = remaining_zero["StockCode"].astype("string")
valid_prices["StockCode"] = valid_prices["StockCode"].astype("string")

# 4. Remove invalid values
remaining_zero = remaining_zero.dropna(
    subset=["InvoiceDate", "StockCode"]
)

valid_prices = valid_prices.dropna(
    subset=["InvoiceDate", "StockCode"]
)

# 5. Rename price columns
valid_prices = valid_prices.rename(columns={
    "InvoiceDate": "Price_Date",
    "UnitPrice": "Historical_Price"
})

# 6. Sort by date
remaining_zero = remaining_zero.sort_values("InvoiceDate")
valid_prices = valid_prices.sort_values("Price_Date")

# 7. Find previous price
previous_prices = pd.merge_asof(
    remaining_zero,
    valid_prices,
    left_on="InvoiceDate",
    right_on="Price_Date",
    by="StockCode",
    direction="backward",
    allow_exact_matches=False
)

# 8. Find next price
next_prices = pd.merge_asof(
    remaining_zero,
    valid_prices,
    left_on="InvoiceDate",
    right_on="Price_Date",
    by="StockCode",
    direction="forward",
    allow_exact_matches=False
)

# 9. Calculate time differences
prev_diff = (
    previous_prices["InvoiceDate"] -
    previous_prices["Price_Date"]
).abs()

next_diff = (
    next_prices["Price_Date"] -
    next_prices["InvoiceDate"]
).abs()

# 10. Choose the nearest price using positional arrays
use_previous = (
    prev_diff.notna() &
    (next_diff.isna() | (prev_diff <= next_diff))
).to_numpy()

prev_values = previous_prices["Historical_Price"].to_numpy()
next_values = next_prices["Historical_Price"].to_numpy()

best_values = np.where(
    use_previous,
    prev_values,
    next_values
)

# 11. Recover prices
target_indices = remaining_zero["_target_index"].to_numpy()

matched = ~pd.isna(best_values)

df_products_clean.loc[
    target_indices[matched],
    "UnitPrice"
] = best_values[matched]

# 12. Results
print("Transactions Checked:", len(remaining_zero))
print("Successfully Recovered:", matched.sum())
print("No Historical Price Found:", (~matched).sum())

print(
    "Remaining Zero Prices:",
    (df_products_clean["UnitPrice"] == 0).sum()
)

Transactions Checked: 1027
Successfully Recovered: 1008
No Historical Price Found: 19
Remaining Zero Prices: 19


In [36]:
# Get the remaining 19 transactions

remaining_zero = df_products_clean[
    df_products_clean["UnitPrice"] == 0
].copy()

# Display all relevant information

display(
    remaining_zero[
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice",
            "CustomerID",
            "InvoiceDate"
        ]
    ].sort_values("StockCode")
)

print("Total Remaining Transactions:", len(remaining_zero))

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,InvoiceDate
19628,537875,20849,Unknown Product,1,0.0,<NA>,2010-12-08 18:08:00
2024,536552,20950,Unknown Product,1,0.0,<NA>,2010-12-01 14:34:00
1970,536545,21134,Unknown Product,1,0.0,<NA>,2010-12-01 14:32:00
254130,559296,23001,TRAVEL CARD WALLET DOTCOMGIFTSHOP,200,0.0,<NA>,2011-07-07 12:35:00
417107,572614,23595,adjustment,5,0.0,<NA>,2011-10-25 11:38:00
346849,567207,35592T,Unknown Product,4,0.0,<NA>,2011-09-19 11:01:00
423946,573169,35600A,Found by jackie,1,0.0,<NA>,2011-10-28 09:56:00
7187,536995,35951,Unknown Product,57,0.0,<NA>,2010-12-03 15:29:00
21786,538137,62095B,Unknown Product,2,0.0,<NA>,2010-12-09 15:57:00
19631,537878,72803B,Unknown Product,1,0.0,<NA>,2010-12-08 18:09:00


Total Remaining Transactions: 19


**19** lines are left without any historical price, and their descriptions are suspicious (`Unknown Product`, `adjustment`, `Found by jackie`...), so I deleted them. This is the only deletion from `df_products_clean`.

In [37]:
# Count before deletion
before = len(df_products_clean)

# Remove remaining zero-price transactions
df_products_clean = df_products_clean[
    df_products_clean["UnitPrice"] != 0
].copy()

# Count after deletion
after = len(df_products_clean)

print("Rows Before:", before)
print("Deleted Rows:", before - after)
print("Rows After:", after)
print(
    "Remaining Zero Prices:",
    (df_products_clean["UnitPrice"] == 0).sum()
)

Rows Before: 537636
Deleted Rows: 19
Rows After: 537617
Remaining Zero Prices: 0


The waterfall summarises the journey: from 1,162 lines with a zero price down to zero, and how much each step recovered.

In [38]:
# Zero-price recovery journey (numbers come from the variables of the two recovery steps)
zero_total = int(zero_mask.sum())            # zero-price product lines before recovery
rec_by_description = int(fillable.sum())     # attempt 1: description with a single known price
rec_by_nearest = int(matched.sum())          # attempt 2: nearest historical price of the StockCode
dropped = zero_total - rec_by_description - rec_by_nearest
remaining = zero_total - rec_by_description - rec_by_nearest - dropped

fig = go.Figure(go.Waterfall(
    measure=["absolute", "relative", "relative", "relative", "total"],
    x=["Zero-price lines", "Recovered: one price per description",
       "Recovered: nearest price per StockCode", "Dropped: no price found", "Remaining"],
    y=[zero_total, -rec_by_description, -rec_by_nearest, -dropped, 0],
    text=[f"{zero_total:,}", f"-{rec_by_description:,}", f"-{rec_by_nearest:,}", f"-{dropped:,}", f"{remaining:,}"],
    textposition="outside",
    increasing=dict(marker=dict(color="#E67E22")),
    decreasing=dict(marker=dict(color="#1B7F4C")),
    totals=dict(marker=dict(color="#2E86DE")),
))
fig.update_layout(title="Zero UnitPrice: from problem to resolution", title_x=0.5,
                  yaxis_title="Product lines", height=450, showlegend=False)
fig.show()

## 6. CustomerID, Duplicates & Feature Preparation

### 6.1 Missing CustomerID

After cleaning, **132,704** lines (24.68%) still have no `CustomerID`. I tried to recover it from the invoice: if an invoice has some lines without a customer and some lines with a known customer, I take the ID from the known lines.  
Result: **0** recoverable lines, because an invoice with a missing customer is missing it on all of its lines. These are invoices of unregistered customers (Guests).

Decision: I created a `Customer_Type` column (Guest / Registered Customer):
- Guests are **removed** from the customer RFM because there is no identity to compute Recency and Frequency on.
- But they **stay** in the product analysis and the EDA because their sales are real.

In [39]:
# Check missing CustomerID in cleaned Products

missing_customer = df_products_clean["CustomerID"].isna()

missing_summary = pd.DataFrame({
    "Count": [
        len(df_products_clean),
        missing_customer.sum(),
        df_products_clean["CustomerID"].notna().sum()
    ],
    "Percentage": [
        100,
        missing_customer.mean() * 100,
        df_products_clean["CustomerID"].notna().mean() * 100
    ]
}, index=[
    "Total Product Transactions",
    "Missing CustomerID",
    "Available CustomerID"
])

display(missing_summary.round(2))

,Count,Percentage
Total Product Transactions,537617,100.00
Missing CustomerID,132704,24.68
Available CustomerID,404913,75.32


In [40]:
# Get sample invoices containing missing CustomerID

missing_invoice_ids = (
    df_products_clean.loc[
        df_products_clean["CustomerID"].isna(),
        "InvoiceNo"
    ]
    .drop_duplicates()
    .head(5)
)

# Display all transactions belonging to these invoices

sample_invoices = df_products_clean[
    df_products_clean["InvoiceNo"].isin(missing_invoice_ids)
].sort_values(
    ["InvoiceNo", "InvoiceDate"]
)

display(
    sample_invoices[
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice",
            "CustomerID",
            "InvoiceDate"
        ]
    ].head(30)
)

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,InvoiceDate
622,536414,22139,RETROSPOT TEA SET CERAMIC 11 PC,56,4.25,<NA>,2010-12-01 11:52:00
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,2.51,<NA>,2010-12-01 14:32:00
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,2.51,<NA>,2010-12-01 14:32:00
1445,536544,21786,POLKADOT RAIN HAT,4,0.85,<NA>,2010-12-01 14:32:00
1446,536544,21787,RAIN PONCHO RETROSPOT,2,1.66,<NA>,2010-12-01 14:32:00
1447,536544,21790,VINTAGE SNAP CARDS,9,1.66,<NA>,2010-12-01 14:32:00
1448,536544,21791,VINTAGE HEADS AND TAILS CARD GAME,2,2.51,<NA>,2010-12-01 14:32:00
1449,536544,21801,CHRISTMAS TREE DECORATION WITH BELL,10,0.43,<NA>,2010-12-01 14:32:00
1450,536544,21802,CHRISTMAS TREE HEART DECORATION,9,0.43,<NA>,2010-12-01 14:32:00
1451,536544,21803,CHRISTMAS TREE STAR DECORATION,11,0.43,<NA>,2010-12-01 14:32:00


In [41]:
# Get unique non-missing CustomerIDs for each Invoice

invoice_customers = (
    df_products_clean.dropna(subset=["CustomerID"])
    .groupby("InvoiceNo")["CustomerID"]
    .agg(lambda x: x.unique())
)

# Keep invoices having exactly one known CustomerID

valid_invoice_customers = invoice_customers[
    invoice_customers.apply(len) == 1
]

# Create Invoice -> CustomerID mapping

invoice_customer_map = {
    invoice: customers[0]
    for invoice, customers in valid_invoice_customers.items()
}

print("Total Invoices with Known Customers:", len(invoice_customers))
print("Invoices with One Unique Customer:", len(invoice_customer_map))

Total Invoices with Known Customers: 21788
Invoices with One Unique Customer: 21788


In [42]:
# Identify missing CustomerID rows

missing_mask = df_products_clean["CustomerID"].isna()

# Check which missing rows have an invoice in our mapping

recoverable_mask = (
    missing_mask &
    df_products_clean["InvoiceNo"].isin(invoice_customer_map)
)

print("Total Missing CustomerID:", missing_mask.sum())

print("Recoverable Transactions:", recoverable_mask.sum())

print(
    "Unrecoverable Transactions:",
    (missing_mask & ~recoverable_mask).sum()
)

print(
    "Recovery Percentage:",
    round(
        recoverable_mask.sum() / missing_mask.sum() * 100, 2
    ) if missing_mask.sum() else 0,
    "%"
)

Total Missing CustomerID: 132704
Recoverable Transactions: 0
Unrecoverable Transactions: 132704
Recovery Percentage: 0.0 %


In [43]:
df_products_clean["Customer_Type"] = np.where(
    df_products_clean["CustomerID"].isna(),
    "Guest",
    "Registered Customer"
)

display(
    df_products_clean["Customer_Type"].value_counts()
)

Customer_Type
Registered Customer    404913
Guest                  132704
Name: count, dtype: int64

These donuts show the weight of Guests in lines, invoices and revenue, in other words how much will be left out of the customer RFM.

In [44]:
# Guest vs Registered (sales lines only)
guest_summary = (
    df_products_clean[df_products_clean["Quantity"] > 0]
    .assign(Revenue=lambda d: d["Quantity"] * d["UnitPrice"])
    .groupby("Customer_Type")
    .agg(Lines=("InvoiceNo", "size"),
         Invoices=("InvoiceNo", "nunique"),
         Revenue=("Revenue", "sum"))
    .reset_index()
)
display(guest_summary)

type_colors = {"Guest": "#E67E22", "Registered Customer": "#2E86DE"}

fig = make_subplots(rows=1, cols=3, specs=[[{"type": "domain"}] * 3],
                    subplot_titles=("Lines", "Invoices", "Revenue"))
for i, metric in enumerate(["Lines", "Invoices", "Revenue"], start=1):
    fig.add_trace(go.Pie(
        labels=guest_summary["Customer_Type"], values=guest_summary[metric],
        hole=0.5, sort=False, name=metric,
        marker=dict(colors=[type_colors[t] for t in guest_summary["Customer_Type"]]),
    ), row=1, col=i)
fig.update_layout(title_text="Guest vs Registered Customers (sales only)", title_x=0.5, height=400)
fig.show()

,Customer_Type,Lines,Invoices,Revenue
0,Guest,132539,2116,1658431.100
1,Registered Customer,396374,18405,8766457.214


In [45]:
df_products_clean.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,LineRevenue,ItemType,Description_Missing,Customer_Type
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,Product,False,Registered Customer
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,Product,False,Registered Customer
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,Product,False,Registered Customer
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,Product,False,Registered Customer
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,Product,False,Registered Customer


### 6.2 Duplicated Transactions

The check found **5,264** duplicate rows (0.98% of the rows). Normally one would delete them, but I **decided not to delete them**, for these reasons:

1. **The data has no Line ID.** A duplicate here means all columns are identical (same invoice, same product, same quantity, same price, same minute). That can be a recording error, but it can also be a customer who took the same item twice in the same invoice (two separate lines for the same product).
2. When I investigated, I found that the duplicates in this dataset are **not necessarily wrong repetitions**, and the repetition happens inside the **same invoice**, so it is not an invoice that was recorded twice.
3. Deleting is irreversible: if I delete a legitimate line I would reduce the customer's Monetary without justification.
4. Their effect on RFM is limited by design: **Frequency** counts unique invoices (`nunique`) and **Recency** takes the last date, so duplicates do not affect them at all. Only **Monetary** can be affected.

Instead of deleting: I measure the size of the problem, create an `Is_Duplicate_Line` column that flags them, and after computing the RFM I run a **sensitivity check** (Section 8.9) that compares the result with and without the duplicates.

In [46]:
# Check duplicated transactions

total_rows = len(df_products_clean)

duplicate_rows = df_products_clean.duplicated().sum()

unique_rows = total_rows - duplicate_rows

duplicate_summary = pd.DataFrame({
    "Count": [
        total_rows,
        duplicate_rows,
        unique_rows
    ],
    "Percentage": [
        100,
        duplicate_rows / total_rows * 100,
        unique_rows / total_rows * 100
    ]
}, index=[
    "Total Product Transactions",
    "Duplicated Transactions",
    "Unique Transactions"
])

display(duplicate_summary.round(2))

,Count,Percentage
Total Product Transactions,537617,100.00
Duplicated Transactions,5264,0.98
Unique Transactions,532353,99.02


In [47]:
# Display duplicated transactions

duplicates_sample = df_products[
    df_products.duplicated(keep=False)
].sort_values(
    by=["InvoiceNo", "StockCode", "InvoiceDate"]
)

display(duplicates_sample.head(30))

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,LineRevenue,ItemType,Description_Missing
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom,1.25,Product,False
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom,1.25,Product,False
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom,4.95,Product,False
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom,4.95,Product,False
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom,2.10,Product,False
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom,2.10,Product,False
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom,2.95,Product,False
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom,2.95,Product,False
565,536412,21448,12 DAISY PEGS IN WOOD BOX,2,2010-12-01 11:49:00,1.65,17920,United Kingdom,3.30,Product,False
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920,United Kingdom,1.65,Product,False


In [48]:
# Measure the duplicates instead of deleting them
dup_mask = df_products_clean.duplicated(keep="first")      # the first copy is NOT flagged
dup_rows = df_products_clean[dup_mask]

row_revenue = df_products_clean["Quantity"] * df_products_clean["UnitPrice"]
is_sale = df_products_clean["Quantity"] > 0
gross_sales = row_revenue[is_sale].sum()
dup_sales = row_revenue[dup_mask & is_sale].sum()

dup_per_invoice = dup_rows.groupby("InvoiceNo").size()
n_invoices = df_products_clean["InvoiceNo"].nunique()

dup_metrics = {
    "Extra duplicate rows": f"{int(dup_mask.sum()):,}",
    "% of all product rows": f"{dup_mask.mean() * 100:.2f}%",
    "Invoices containing duplicates": f"{dup_per_invoice.size:,}",
    "% of all invoices": f"{dup_per_invoice.size / n_invoices * 100:.2f}%",
    "Max extra copies inside one invoice": f"{int(dup_per_invoice.max()):,}",
    "Revenue carried by duplicate copies": f"{dup_sales:,.0f}",
    "% of gross sales revenue": f"{dup_sales / gross_sales * 100:.2f}%",
    "Duplicates inside cancellation invoices": f"{int(dup_rows['InvoiceNo'].astype(str).str.startswith('C').sum()):,}",
    "Duplicates belonging to Guests": f"{int((dup_rows['Customer_Type'] == 'Guest').sum()):,}",
}
display(pd.Series(dup_metrics, name="Value").to_frame())

# How many extra copies does an invoice usually carry?
dup_dist = dup_per_invoice.clip(upper=10).value_counts().sort_index().reset_index()
dup_dist.columns = ["Extra_Copies", "Invoices"]
dup_dist["Extra_Copies"] = dup_dist["Extra_Copies"].astype(str).replace({"10": "10+"})

fig = px.bar(
    dup_dist, x="Extra_Copies", y="Invoices", text="Invoices",
    title="Duplicates happen INSIDE invoices: extra copies per invoice",
    labels={"Extra_Copies": "Extra duplicate copies inside one invoice", "Invoices": "Number of invoices"},
    color_discrete_sequence=["#2E86DE"],
)
fig.update_layout(title_x=0.5, height=400)
fig.show()

,Value
Extra duplicate rows,"5,264"
% of all product rows,0.98%
Invoices containing duplicates,"1,933"
% of all invoices,8.07%
Max extra copies inside one invoice,34
Revenue carried by duplicate copies,"24,792"
% of gross sales revenue,0.24%
Duplicates inside cancellation invoices,36
Duplicates belonging to Guests,42


The decision is now documented: the duplicates remain in the data but are flagged, so I can remove them at any time or compare against them.

In [49]:
# Flag, don't delete
df_products_clean["Is_Duplicate_Line"] = dup_mask
print("Flagged duplicate lines:", int(df_products_clean["Is_Duplicate_Line"].sum()))
print("Rows still in the data:", len(df_products_clean))

Flagged duplicate lines: 5264
Rows still in the data: 537617


### 6.3 Returns (Negative Quantity)

After separating the Inventory Adjustments, all the remaining negative quantities are **cancellations** (customer returns): **8,704** lines, about **1.6%** of the data. I keep them in `df_products_clean` so that returns can be analysed on their own (section 7.9), but I **do not include them in the RFM**: RFM measures buying, and returns are analysed separately.

In [50]:
# Analyze negative and positive quantities

quantity_analysis = pd.DataFrame({
    "Count": [
        (df_products_clean["Quantity"] < 0).sum(),
        (df_products_clean["Quantity"] == 0).sum(),
        (df_products_clean["Quantity"] > 0).sum()
    ]
}, index=[
    "Negative Quantity",
    "Zero Quantity",
    "Positive Quantity"
])

quantity_analysis["Percentage"] = (
    quantity_analysis["Count"] / len(df_products) * 100
)

display(quantity_analysis.round(2))

,Count,Percentage
Negative Quantity,8704,1.62
Zero Quantity,0,0.00
Positive Quantity,528913,98.38


In [51]:
# Display sample negative quantity transactions

negative_returns = df_products[
    df_products["Quantity"] < 0
].copy()

display(
    negative_returns[
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice",
            "LineRevenue",
            "CustomerID",
            "InvoiceDate"
        ]
    ].head(20)
)

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,LineRevenue,CustomerID,InvoiceDate
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,4.65,-4.65,15311,2010-12-01 09:49:00
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,1.65,-19.80,17548,2010-12-01 10:24:00
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,0.29,-6.96,17548,2010-12-01 10:24:00
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,0.29,-6.96,17548,2010-12-01 10:24:00
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,0.29,-6.96,17548,2010-12-01 10:24:00
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,3.45,-41.40,17548,2010-12-01 10:24:00
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,1.65,-19.80,17548,2010-12-01 10:24:00
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,1.65,-39.60,17548,2010-12-01 10:24:00
939,C536506,22960,JAM MAKING SET WITH JARS,-6,4.25,-25.50,17897,2010-12-01 12:38:00
1441,C536543,22632,HAND WARMER RED RETROSPOT,-1,2.10,-2.10,17841,2010-12-01 14:30:00


### 6.4 Revenue Recalculation & Date Features

Prices changed after recovering the zero prices, so the old `LineRevenue` became stale. I dropped it and recomputed `TotalLineRevenue = Quantity × UnitPrice`, which is the one used for everything from here on. Then I converted `InvoiceDate` to datetime and extracted features from it (Year, Month, Day_Name, Hour, ...).

In [52]:
# Recompute revenue AFTER price recovery (old LineRevenue is stale)
df_products_clean = df_products_clean.drop(columns="LineRevenue")
df_products_clean["TotalLineRevenue"] = (
    df_products_clean["Quantity"] * df_products_clean["UnitPrice"]
)

In [53]:
# Convert InvoiceDate to datetime
df_products_clean["InvoiceDate"] = pd.to_datetime(
    df_products_clean["InvoiceDate"],
    errors="coerce"
)

# Extract date and time features
df_products_clean["Year"] = df_products_clean["InvoiceDate"].dt.year
df_products_clean["Month"] = df_products_clean["InvoiceDate"].dt.month
df_products_clean["Month_Name"] = df_products_clean["InvoiceDate"].dt.month_name()
df_products_clean["Day"] = df_products_clean["InvoiceDate"].dt.day
df_products_clean["Day_Name"] = df_products_clean["InvoiceDate"].dt.day_name()
df_products_clean["Hour"] = df_products_clean["InvoiceDate"].dt.hour
df_products_clean["DayOfWeek"] = df_products_clean["InvoiceDate"].dt.dayofweek

# Display sample
display(
    df_products_clean[
        [
            "InvoiceDate",
            "Year",
            "Month",
            "Month_Name",
            "Day",
            "Day_Name",
            "Hour",
            "DayOfWeek"
        ]
    ].head(10)
)

C:\Users\hamas\AppData\Local\Temp\ipykernel_22640\2095026839.py:14: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df_products_clean["DayOfWeek"] = df_products_clean["InvoiceDate"].dt.dayofweek


,InvoiceDate,Year,Month,Month_Name,Day,Day_Name,Hour,DayOfWeek
0,2010-12-01 08:26:00,2010,12,December,1,Wednesday,8,2
1,2010-12-01 08:26:00,2010,12,December,1,Wednesday,8,2
2,2010-12-01 08:26:00,2010,12,December,1,Wednesday,8,2
3,2010-12-01 08:26:00,2010,12,December,1,Wednesday,8,2
4,2010-12-01 08:26:00,2010,12,December,1,Wednesday,8,2
5,2010-12-01 08:26:00,2010,12,December,1,Wednesday,8,2
6,2010-12-01 08:26:00,2010,12,December,1,Wednesday,8,2
7,2010-12-01 08:28:00,2010,12,December,1,Wednesday,8,2
8,2010-12-01 08:28:00,2010,12,December,1,Wednesday,8,2
9,2010-12-01 08:34:00,2010,12,December,1,Wednesday,8,2


In [54]:
print("Start Date:", df_products_clean["InvoiceDate"].min())
print("End Date:", df_products_clean["InvoiceDate"].max())

print("Missing Dates:", df_products_clean["InvoiceDate"].isna().sum())

print("Years:", sorted(df_products_clean["Year"].dropna().unique()))

Start Date: 2010-12-01 08:26:00
End Date: 2011-12-09 12:50:00
Missing Dates: 0
Years: [np.int32(2010), np.int32(2011)]


In [55]:
df_products_clean.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,ItemType,Description_Missing,Customer_Type,Is_Duplicate_Line,TotalLineRevenue,Year,Month,Month_Name,Day,Day_Name,Hour,DayOfWeek
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,Product,False,Registered Customer,False,15.30,2010,12,December,1,Wednesday,8,2
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Product,False,Registered Customer,False,20.34,2010,12,December,1,Wednesday,8,2
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,Product,False,Registered Customer,False,22.00,2010,12,December,1,Wednesday,8,2
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Product,False,Registered Customer,False,20.34,2010,12,December,1,Wednesday,8,2
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Product,False,Registered Customer,False,20.34,2010,12,December,1,Wednesday,8,2


### 6.5 Hourly Activity & Sales vs Returns

I split the data into sales (`Quantity > 0`) and returns (`Quantity < 0`) and compare months and hours. What stood out:
- By number of invoices, activity peaks between **10:00 and 15:00**, and the busiest hour is **12**.
- At 6 AM there are 38 return lines against a single sales line, so returns are recorded at different times than sales.

In [56]:
hourly_transactions = (
    df_products_clean
    .groupby("Hour")
    .agg(
        Total_Transactions=("InvoiceNo", "count"),
        Unique_Invoices=("InvoiceNo", "nunique")
    )
)

display(hourly_transactions)

,Total_Transactions,Unique_Invoices
Hour,,
6,39,20
7,379,31
8,8841,618
9,34050,1737
10,48549,2764
11,57202,2917
12,78183,3716
13,71803,3186
14,66912,2856


In [57]:
# Split Sales and Returns
sales = df_products_clean[df_products_clean["Quantity"] > 0].copy()
returns = df_products_clean[df_products_clean["Quantity"] < 0].copy()

# Monthly Analysis
sales_monthly = sales.groupby(
    sales["InvoiceDate"].dt.to_period("M")
).agg(
    Sales_Transactions=("InvoiceNo", "count"),
    Sales_Invoices=("InvoiceNo", "nunique"),
    Sold_Quantity=("Quantity", "sum")
)

returns_monthly = returns.groupby(
    returns["InvoiceDate"].dt.to_period("M")
).agg(
    Return_Transactions=("InvoiceNo", "count"),
    Return_Invoices=("InvoiceNo", "nunique"),
    Returned_Quantity=("Quantity", lambda x: abs(x.sum()))
)

monthly_comparison = sales_monthly.join(
    returns_monthly, how="outer"
).fillna(0)

monthly_comparison.index = monthly_comparison.index.astype(str)

display(monthly_comparison)

,Sales_Transactions,Sales_Invoices,Sold_Quantity,Return_Transactions,Return_Invoices,Returned_Quantity
InvoiceDate,,,,,,
2010-12,41508,1609,361734,698,301,16009
2011-01,34191,1114,396355,679,247,79784
2011-02,27042,1120,285101,418,191,2606
2011-03,35718,1517,383533,642,281,4338
2011-04,29033,1307,311875,531,220,13600
2011-05,36106,1719,398277,558,267,4057
2011-06,35853,1568,393942,669,299,7215
2011-07,38515,1517,406938,642,236,5946
2011-08,34405,1385,424629,617,240,11662


In [58]:
# Hourly Sales vs Returns

sales_hourly = sales.groupby("Hour").agg(
    Sales_Transactions=("InvoiceNo", "count"),
    Sold_Quantity=("Quantity", "sum")
)

returns_hourly = returns.groupby("Hour").agg(
    Return_Transactions=("InvoiceNo", "count"),
    Returned_Quantity=("Quantity", lambda x: abs(x.sum()))
)

hourly_comparison = sales_hourly.join(
    returns_hourly, how="outer"
).fillna(0)

display(hourly_comparison.sort_index())

,Sales_Transactions,Sold_Quantity,Return_Transactions,Returned_Quantity
Hour,,,,
6,1,1,38,85
7,375,15360,4,9
8,8742,158963,99,953
9,33548,520286,502,86190
10,47585,816740,964,89037
11,56092,677231,1110,18611
12,76968,851878,1215,9369
13,70799,708085,1004,17664
14,65993,605140,919,20791


In [59]:
monthly_plot = (
    monthly_comparison[["Sales_Transactions", "Return_Transactions"]]
    .rename_axis("Month")
    .reset_index()
    .melt(id_vars="Month", var_name="Type", value_name="Transactions")
)

fig = px.bar(
    monthly_plot, x="Month", y="Transactions", color="Type", barmode="group",
    title="Monthly Sales vs Returns (transaction lines)",
    color_discrete_map={"Sales_Transactions": "#2E86DE", "Return_Transactions": "#E67E22"},
)
fig.update_layout(title_x=0.5, xaxis_tickangle=-45, legend_title="", height=450)
fig.show()

I created `Is_UK` because about **91.6%** of the rows are from the UK, which will help in the market analysis.

In [60]:

# UK vs International flag
df_products_clean["Is_UK"] = df_products_clean["Country"].eq("United Kingdom")

print("Rows now:", len(df_products_clean))
print(df_products_clean["Is_UK"].value_counts(normalize=True).round(3))

Rows now: 537617
Is_UK
True     0.916
False    0.084
Name: proportion, dtype: float64


## 7. Interactive Exploratory Data Analysis (EDA) with Plotly

In this part the time-based and commercial analyses are turned into interactive visualisations using **Plotly Express** and **Plotly Graph Objects**.  
Every chart offers hover details, zoom and legend filtering, which helps us explore the details instead of settling for a static picture.

> **Analytical note:** Sales and Returns are kept separate. The sales, product and customer indicators in the following charts are computed from transactions with `Quantity > 0`, while returns appear as a separate indicator. This prevents mixing the value of positive sales with returns while we try to understand buying behaviour.

In [61]:
# %pip install -U plotly

In [62]:

import plotly.express as px
import plotly.graph_objects as go

# Rebuild consistent analysis subsets
eda_sales = df_products_clean.loc[df_products_clean["Quantity"] > 0].copy()
eda_returns = df_products_clean.loc[df_products_clean["Quantity"] < 0].copy()

# Keep only rows with a valid date for time-based analysis
eda_sales = eda_sales.dropna(subset=["InvoiceDate"])
eda_returns = eda_returns.dropna(subset=["InvoiceDate"])

# Ensure the revenue field is up to date
eda_sales["TotalLineRevenue"] = eda_sales["Quantity"] * eda_sales["UnitPrice"]
eda_returns["TotalLineRevenue"] = eda_returns["Quantity"] * eda_returns["UnitPrice"]

print(f"Sales rows: {len(eda_sales):,}")
print(f"Return rows: {len(eda_returns):,}")
print(f"Customers with IDs: {eda_sales['CustomerID'].nunique():,}")

Sales rows: 528,913
Return rows: 8,704
Customers with IDs: 4,335


### 7.1 Monthly Sales and Returns

**Business question:** Do sales and returns rise or fall in particular months?  
We show the number of invoices and the sales value over time, together with the number of return invoices in the same chart.

In [63]:
monthly_sales_eda = (
    eda_sales.assign(MonthStart=eda_sales["InvoiceDate"].dt.to_period("M").dt.to_timestamp())
    .groupby("MonthStart")
    .agg(
        Revenue=("TotalLineRevenue", "sum"),
        Sales_Invoices=("InvoiceNo", "nunique"),
        Units_Sold=("Quantity", "sum")
    )
    .reset_index()
)

monthly_returns_eda = (
    eda_returns.assign(MonthStart=eda_returns["InvoiceDate"].dt.to_period("M").dt.to_timestamp())
    .groupby("MonthStart")
    .agg(Return_Invoices=("InvoiceNo", "nunique"))
    .reset_index()
)

monthly_eda = monthly_sales_eda.merge(monthly_returns_eda, on="MonthStart", how="left")
monthly_eda["Return_Invoices"] = monthly_eda["Return_Invoices"].fillna(0)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=monthly_eda["MonthStart"], y=monthly_eda["Revenue"],
    mode="lines+markers", name="Sales Revenue",
    line=dict(width=3), yaxis="y"
))
fig.add_trace(go.Bar(
    x=monthly_eda["MonthStart"], y=monthly_eda["Sales_Invoices"],
    name="Sales Invoices", opacity=0.45, yaxis="y2"
))
fig.add_trace(go.Scatter(
    x=monthly_eda["MonthStart"], y=monthly_eda["Return_Invoices"],
    mode="lines+markers", name="Return Invoices",
    line=dict(dash="dash"), yaxis="y2"
))
fig.update_layout(
    title="Monthly Revenue, Sales Invoices and Return Invoices",
    xaxis_title="Month",
    yaxis=dict(title="Revenue"),
    yaxis2=dict(title="Number of Invoices", overlaying="y", side="right"),
    hovermode="x unified",
    legend_title="Metric",
    template="plotly_white"
)
fig.show()

### 7.2 Sales Activity by Hour

**Business question:** At which hours is buying activity the highest?  
The result can help us understand peak times and plan customer service and operations.

In [64]:
hourly_eda = (
    eda_sales.groupby("Hour")
    .agg(
        Sales_Invoices=("InvoiceNo", "nunique"),
        Transactions=("InvoiceNo", "size"),
        Revenue=("TotalLineRevenue", "sum")
    )
    .reset_index()
    .sort_values("Hour")
)

fig = px.bar(
    hourly_eda, x="Hour", y="Sales_Invoices",
    hover_data=["Transactions", "Revenue"],
    title="Sales Activity by Hour",
    labels={"Sales_Invoices": "Unique Sales Invoices", "Hour": "Hour of Day"},
    template="plotly_white"
)
fig.update_traces(marker_line_width=0)
fig.update_xaxes(dtick=1)
fig.show()

### 7.3 Sales by Day of Week

**Business question:** Are there days of the week with more invoices or more revenue than others?

In [65]:
weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

weekday_eda = (
    eda_sales.groupby("Day_Name")
    .agg(
        Sales_Invoices=("InvoiceNo", "nunique"),
        Revenue=("TotalLineRevenue", "sum")
    )
    .reindex(weekday_order)
    .reset_index()
)

fig = px.bar(
    weekday_eda,
    x="Day_Name",
    y="Sales_Invoices",
    color="Revenue",
    color_continuous_scale=[
        [0, "#9ECAE1"],
        [0.25, "#6BAED6"],
        [0.5, "#3182BD"],
        [0.75, "#08519C"],
        [1, "#08306B"]
    ],
    hover_data={
        "Revenue": ":,.2f",
        "Sales_Invoices": True
    },
    title="Sales Invoices by Day of Week",
    labels={
        "Day_Name": "Day",
        "Sales_Invoices": "Unique Sales Invoices",
        "Revenue": "Total Revenue"
    },
    template="plotly_white"
)

fig.update_layout(
    title_x=0.5,
    title_font_size=20,
    xaxis_title="Day of Week",
    yaxis_title="Unique Sales Invoices",
    coloraxis_colorbar_title="Revenue",
    font=dict(family="Arial", size=12)
)

fig.update_traces(
    marker_line_color="#17365D",
    marker_line_width=0.7
)

fig.show()

### 7.4 Top Products by Revenue and Quantity

**Business question:** Which products contribute the most to revenue? And are they the same as the best sellers by quantity?  
We use `StockCode` as the product identifier and add the description when it is available.

In [66]:
product_eda = (
    eda_sales.groupby("StockCode")
    .agg(
        Revenue=("TotalLineRevenue", "sum"),
        Units_Sold=("Quantity", "sum"),
        Invoices=("InvoiceNo", "nunique")
    )
    .reset_index()
)

if "Description" in eda_sales.columns:
    product_names = (
        eda_sales.dropna(subset=["Description"])
        .groupby("StockCode")["Description"]
        .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0])
        .rename("Description")
        .reset_index()
    )
    product_eda = product_eda.merge(product_names, on="StockCode", how="left")
    product_eda["Product_Label"] = (
        product_eda["StockCode"].astype(str) + " - " +
        product_eda["Description"].fillna("No description").astype(str).str.slice(0, 35)
    )
else:
    product_eda["Product_Label"] = product_eda["StockCode"].astype(str)

top_products_revenue = product_eda.nlargest(15, "Revenue").sort_values("Revenue")
fig = px.bar(
    top_products_revenue, x="Revenue", y="Product_Label",
    orientation="h", hover_data=["StockCode", "Units_Sold", "Invoices"],
    title="Top 15 Products by Revenue",
    labels={"Revenue": "Revenue", "Product_Label": "Product"},
    template="plotly_white"
)
fig.show()

top_products_units = product_eda.nlargest(15, "Units_Sold").sort_values("Units_Sold")
fig = px.bar(
    top_products_units, x="Units_Sold", y="Product_Label",
    orientation="h", hover_data=["StockCode", "Revenue", "Invoices"],
    title="Top 15 Products by Units Sold",
    labels={"Units_Sold": "Units Sold", "Product_Label": "Product"},
    template="plotly_white"
)
fig.show()

### 7.5 Country Performance

**Business question:** Which countries have the highest revenue and the most customers?  
Rows without a Country are excluded, and only registered customers are used when counting customers.

**Filters.** The bar chart has two dropdown menus above it:
- **View** — top 15 countries, top 15 without the United Kingdom, all countries (with or without the UK), or **one single country** (`Only <country>`). Bar colour = number of unique customers; hover shows revenue, invoices and customers.
- **Axis** — *Linear* shows the real proportions; *Log* makes small countries visible even though the United Kingdom is about 85% of revenue.

The small helper `add_dropdown` is defined at the top of the cell below (it is re-declared in later sections that use it).

In [67]:
def add_dropdown(fig, buttons, x=0.0, y=1.0):
    """Attach one more dropdown menu above the plot (keeps any menus already on the figure)."""
    menu = dict(type="dropdown", buttons=buttons, x=x, xanchor="left", y=y, yanchor="bottom",
                showactive=True, bgcolor="white", bordercolor="#BDC3C7")
    fig.update_layout(updatemenus=list(fig.layout.updatemenus) + [menu], margin=dict(t=130))

country_eda = (
    eda_sales.dropna(subset=["Country"])
    .groupby("Country")
    .agg(
        Revenue=("TotalLineRevenue", "sum"),
        Sales_Invoices=("InvoiceNo", "nunique"),
        Customers=("CustomerID", "nunique")
    )
    .reset_index()
)

# ---- Interactive country filter -------------------------------------------------
# One bar trace; the dropdown swaps its data (restyle), so any view can be selected.
def country_view(df):
    d = df.sort_values("Revenue")                       # ascending -> biggest bar on top
    return dict(x=d["Revenue"].tolist(), y=d["Country"].tolist(),
                marker_color=d["Customers"].tolist(),
                customdata=d[["Sales_Invoices", "Customers"]].values.tolist())

by_revenue = country_eda.sort_values("Revenue", ascending=False)
no_uk = by_revenue[by_revenue["Country"] != "United Kingdom"]

views = {
    "Top 15 countries":                    by_revenue.head(15),
    "Top 15 - hide United Kingdom":        no_uk.head(15),
    "All countries":                       by_revenue,
    "All countries - hide United Kingdom": no_uk,
}
for country in by_revenue["Country"]:                  # one entry per country (biggest first)
    views[f"Only {country}"] = by_revenue[by_revenue["Country"] == country]

first = country_view(views["Top 15 countries"])
fig = go.Figure(go.Bar(
    x=first["x"], y=first["y"], orientation="h", customdata=first["customdata"],
    marker=dict(color=first["marker_color"], colorscale="Viridis", showscale=True,
                cmin=float(country_eda["Customers"].min()), cmax=float(country_eda["Customers"].max()),
                colorbar=dict(title="Unique Customers")),
    hovertemplate="%{y}<br>Revenue: %{x:,.0f}<br>Invoices: %{customdata[0]:,}<br>Customers: %{customdata[1]:,}<extra></extra>",
))
fig.update_yaxes(type="category")
fig.update_layout(title="Sales Revenue by Country", title_x=0.5, height=560,
                  xaxis_title="Revenue", template="plotly_white")

def view_button(name, df):
    v = country_view(df)
    return dict(label=name, method="update",
                args=[{"x": [v["x"]], "y": [v["y"]], "marker.color": [v["marker_color"]], "customdata": [v["customdata"]]},
                      {"title.text": f"Sales Revenue by Country: {name}", "height": max(420, 28 * len(df) + 160)}])

add_dropdown(fig, [view_button(name, df) for name, df in views.items()], x=0.0)
add_dropdown(fig, [
    dict(label="Linear axis", method="relayout", args=[{"xaxis.type": "linear", "xaxis.autorange": True}]),
    dict(label="Log axis (see small countries)", method="relayout", args=[{"xaxis.type": "log", "xaxis.autorange": True}]),
], x=0.3)
fig.show()

### 7.6 Customer Contribution

**Business question:** Does revenue depend on a small number of customers?  
The following analysis covers only customers who have a `CustomerID`, because we cannot build a reliable customer-level profile for Guest transactions.

In [68]:
customer_eda = (
    eda_sales.dropna(subset=["CustomerID"])
    .groupby("CustomerID")
    .agg(
        Monetary=("TotalLineRevenue", "sum"),
        Frequency=("InvoiceNo", "nunique"),
        Units_Bought=("Quantity", "sum")
    )
    .reset_index()
)

top_customers = customer_eda.nlargest(20, "Monetary").sort_values("Monetary")
fig = px.bar(
    top_customers, x="Monetary", y="CustomerID",
    orientation="h", color="Frequency",
    hover_data=["Units_Bought"],
    title="Top 20 Customers by Revenue",
    labels={"Monetary": "Revenue", "CustomerID": "Customer ID",
            "Frequency": "Unique Invoices"},
    template="plotly_white"
)
fig.update_yaxes(type="category")
fig.show()

### 7.7 Basket Size and Average Order Value (AOV)

**Business question:** What is the typical basket size? And does the value of a single invoice point to a difference between individual purchases and wholesale purchases?  
Here we compute the total quantity and the sales value of every invoice, using positive sales only.

In [69]:
invoice_eda = (
    eda_sales.groupby("InvoiceNo")
    .agg(
        Basket_Units=("Quantity", "sum"),
        Order_Value=("TotalLineRevenue", "sum"),
        CustomerID=("CustomerID", "first")
    )
    .reset_index()
)

aov = invoice_eda["Order_Value"].mean()
median_order_value = invoice_eda["Order_Value"].median()

print(f"Average Order Value (AOV): {aov:,.2f}")
print(f"Median Order Value: {median_order_value:,.2f}")
print(f"Average Basket Units: {invoice_eda['Basket_Units'].mean():,.2f}")

# Order values are heavily right-skewed -> plot on a log10 scale so the shape is visible
pos = invoice_eda[(invoice_eda["Order_Value"] > 0) & (invoice_eda["Basket_Units"] > 0)].copy()
pos["log10_Order_Value"] = np.log10(pos["Order_Value"])

tick_vals = [10, 30, 100, 300, 1_000, 3_000, 10_000, 30_000, 100_000]
tick_vals = [v for v in tick_vals if pos["Order_Value"].min() / 2 <= v <= pos["Order_Value"].max() * 2]

fig = px.histogram(
    pos, x="log10_Order_Value", nbins=60,
    title="Distribution of Order Values (log scale)",
    labels={"log10_Order_Value": "Order Value (log scale)"},
    template="plotly_white", color_discrete_sequence=["#2E86DE"]
)
# Two labels were overlapping before: put them on opposite sides of their lines
fig.add_vline(x=np.log10(median_order_value), line_dash="dot", line_color="#C0392B",
              annotation_text=f"Median = {median_order_value:,.0f}", annotation_position="top left")
fig.add_vline(x=np.log10(aov), line_dash="dash", line_color="#2C3E50",
              annotation_text=f"Mean = {aov:,.0f}", annotation_position="top right")
fig.update_xaxes(tickvals=np.log10(tick_vals), ticktext=[f"{v:,}" for v in tick_vals])
fig.update_yaxes(title_text="Number of invoices")
fig.update_layout(title_x=0.5, height=430, bargap=0.05)
fig.show()

# Same idea for the scatter: log-log axes so two huge invoices don't squash everyone else
fig = px.scatter(
    pos, x="Basket_Units", y="Order_Value", log_x=True, log_y=True,
    title="Basket Size vs Order Value (log-log)",
    labels={"Basket_Units": "Units per Invoice", "Order_Value": "Order Value"},
    hover_data=["InvoiceNo", "CustomerID"],
    opacity=0.45, template="plotly_white", color_discrete_sequence=["#2E86DE"]
)
fig.update_layout(title_x=0.5, height=480)
fig.show()


Average Order Value (AOV): 508.01
Median Order Value: 298.44
Average Basket Units: 275.01


### 7.8 Product Revenue Concentration — Pareto Analysis

**Business question:** Is a small share of the products responsible for a large share of the revenue?  
We sort the products by revenue in descending order and compute the cumulative share, then compare it with the 80/20 rule as a hypothesis we test, not as an assumed result.

In [70]:
pareto_products = product_eda.loc[product_eda["Revenue"] > 0].copy()
pareto_products = pareto_products.sort_values("Revenue", ascending=False).reset_index(drop=True)

total_product_revenue = pareto_products["Revenue"].sum()
pareto_products["Revenue_Share"] = pareto_products["Revenue"] / total_product_revenue
pareto_products["Cumulative_Revenue_Share"] = pareto_products["Revenue_Share"].cumsum()
pareto_products["Cumulative_Product_Share"] = (
    np.arange(1, len(pareto_products) + 1) / len(pareto_products)
)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=pareto_products["Cumulative_Product_Share"],
    y=pareto_products["Revenue_Share"],
    name="Revenue share per product",
    opacity=0.35
))
fig.add_trace(go.Scatter(
    x=pareto_products["Cumulative_Product_Share"],
    y=pareto_products["Cumulative_Revenue_Share"],
    mode="lines", name="Cumulative revenue share",
    line=dict(width=3)
))
fig.add_hline(y=0.80, line_dash="dash", annotation_text="80% Revenue")
fig.add_vline(x=0.20, line_dash="dot", annotation_text="20% Products")
fig.update_layout(
    title="Pareto Analysis: Cumulative Product Revenue",
    xaxis_title="Cumulative Share of Products",
    yaxis_title="Share of Revenue",
    xaxis=dict(tickformat=".0%"),
    yaxis=dict(tickformat=".0%"),
    hovermode="x unified",
    template="plotly_white"
)
fig.show()

products_for_80 = (
    pareto_products["Cumulative_Revenue_Share"].ge(0.80).idxmax() + 1
    if not pareto_products.empty else 0
)
share_products_for_80 = products_for_80 / len(pareto_products) if len(pareto_products) else np.nan
print(f"Products needed to reach 80% of revenue: {products_for_80:,}")
print(f"Share of products needed: {share_products_for_80:.1%}" if pd.notna(share_products_for_80) else "No positive product revenue available.")

Products needed to reach 80% of revenue: 830
Share of products needed: 21.2%


In [71]:
# Key numbers behind the EDA charts (so the conclusions below rest on real output)
top_month = monthly_eda.loc[monthly_eda["Revenue"].idxmax()]
print(f"Highest revenue month         : {top_month['MonthStart']:%B %Y} -> {top_month['Revenue']:,.0f}")

best_hour = hourly_eda.loc[hourly_eda["Sales_Invoices"].idxmax()]
print(f"Busiest hour                  : {int(best_hour['Hour']):02d}:00 -> {int(best_hour['Sales_Invoices']):,} invoices")

best_day = weekday_eda.loc[weekday_eda["Sales_Invoices"].idxmax()]
print(f"Busiest weekday               : {best_day['Day_Name']} -> {int(best_day['Sales_Invoices']):,} invoices")
no_sales_days = weekday_eda.loc[weekday_eda["Sales_Invoices"].isna(), "Day_Name"].tolist()
print(f"Weekdays with no sales        : {no_sales_days if no_sales_days else 'none'}")

uk_share = country_eda.loc[country_eda["Country"] == "United Kingdom", "Revenue"].sum() / country_eda["Revenue"].sum()
print(f"UK share of sales revenue     : {uk_share:.1%}")
print("Top 5 countries by revenue    :", ", ".join(country_eda.nlargest(5, "Revenue")["Country"]))

top10_rev = set(product_eda.nlargest(10, "Revenue")["StockCode"])
top10_units = set(product_eda.nlargest(10, "Units_Sold")["StockCode"])
print(f"Top-10 overlap (revenue/units): {len(top10_rev & top10_units)} of 10 products")

cust_rank = customer_eda.sort_values("Monetary", ascending=False)
n_top10 = int(np.ceil(len(cust_rank) * 0.10))
print(f"Top 10% customers ({n_top10:,})      : {cust_rank['Monetary'].head(n_top10).sum() / cust_rank['Monetary'].sum():.1%} of registered-customer revenue")

guest_share = eda_sales.loc[eda_sales["CustomerID"].isna(), "TotalLineRevenue"].sum() / eda_sales["TotalLineRevenue"].sum()
print(f"Guest share of sales revenue  : {guest_share:.1%}")
print(f"Order value mean vs median    : {aov:,.2f} vs {median_order_value:,.2f}")

Highest revenue month         : November 2011 -> 1,470,877
Busiest hour                  : 12:00 -> 3,307 invoices
Busiest weekday               : Thursday -> 4,363 invoices
Weekdays with no sales        : ['Saturday']
UK share of sales revenue     : 85.4%
Top 5 countries by revenue    : United Kingdom, Netherlands, EIRE, Germany, France
Top-10 overlap (revenue/units): 6 of 10 products
Top 10% customers (434)      : 61.3% of registered-customer revenue
Guest share of sales revenue  : 15.9%
Order value mean vs median    : 508.01 vs 298.44


### 7.9 Returns Analysis

So far returns were only counted (lines per month and per hour). This section analyses them properly: **how much money they represent, when they happen, which products, countries and customers they come from, and how long after the purchase they happen.** It also prepares a per-customer returns table that can later be used to compute a *net* Monetary value for the RFM.

**What is a "return" here?** A row with `Quantity < 0` that belongs to a **cancellation invoice** (invoice number starting with `C`). The 1,336 inventory-adjustment lines (damaged / thrown-away stock, no customer, price 0) were already separated in section 3, so what remains are genuine customer returns / cancellations.

**Definitions used below**

| Term | Meaning |
|---|---|
| **Return Value** | The absolute value of the returned lines: `-(Quantity × UnitPrice)`, so it is a positive number. |
| **Return Units** | The absolute returned quantity. |
| **Return Rate (%)** | `Return Value ÷ Sales Value × 100` for the same slice (a month, a product, a country, a customer). |

> **Important limitation.** A cancellation invoice does not reference the invoice it cancels, so a return **cannot be tied to its original sale**. The return rate is therefore an *approximation*: a ratio of two totals. A return may belong to a sale made in an earlier month, or even before the dataset starts, so for small slices the rate can exceed 100%.

**Questions answered in this section**

1. How big are returns in lines, invoices, units and money, compared with sales? (7.9.2)
2. Are a few extreme lines distorting the picture? (7.9.3)
3. Is there a pattern over time? (7.9.4)
4. Which products and countries return the most? (7.9.5 – 7.9.6)
5. Which customers return the most, and are returns concentrated in a few of them? (7.9.7 – 7.9.8)
6. How long after the purchase do returns happen, and can they be matched to an earlier purchase at all? (7.9.9)

**About the filters.** As in section 8, a few very large values dominate several charts, so the interactive charts have **dropdown menus** to exclude the extreme lines, hide the United Kingdom, switch the metric or change the axis scale.

In [72]:
# Helpers for the dropdown filters (the same idea is re-used in section 8)
def add_dropdown(fig, buttons, x=0.0, y=1.0):
    """Attach one more dropdown menu above the plot (keeps any menus already on the figure)."""
    menu = dict(type="dropdown", buttons=buttons, x=x, xanchor="left", y=y, yanchor="bottom",
                showactive=True, bgcolor="white", bordercolor="#BDC3C7")
    fig.update_layout(updatemenus=list(fig.layout.updatemenus) + [menu], margin=dict(t=130))


def visibility_buttons(fig, groups, always=()):
    """Buttons: All / Only <group>. Works on figures with one trace per group (trace name = group name).
    Traces listed in `always` (e.g. a reference line) stay visible whatever is selected."""
    names = [t.name for t in fig.data]
    def visible(keep):
        return [(n in keep) or (n in always) for n in names]
    buttons = [dict(label="All", method="update", args=[{"visible": visible(set(groups))}])]
    buttons += [dict(label=f"Only {g}", method="update", args=[{"visible": visible({g})}]) for g in groups]
    return buttons

#### 7.9.1 Building the Returns and Sales Tables

Two tables are built from `df_products_clean`:

- **`returns_df`** — cancellation lines (`Quantity < 0` and invoice starts with `C`), with two helper columns: `Return_Units` and `Return_Value` (both positive) and the month of the return.
- **`sales_df`** — valid sales lines (`Quantity > 0`, `UnitPrice > 0`, not a cancellation). This is the same definition the RFM uses, so returns are always compared with the same sales base.

The sanity check below confirms that the split is clean: there should be **no** negative-quantity lines outside cancellation invoices, and no positive-quantity lines inside them.

In [73]:
is_cancel_invoice = df_products_clean["InvoiceNo"].astype(str).str.startswith("C")
has_date = df_products_clean["InvoiceDate"].notna()

returns_df = df_products_clean[(df_products_clean["Quantity"] < 0) & is_cancel_invoice & has_date].copy()
returns_df["Return_Units"] = -returns_df["Quantity"]
returns_df["Return_Value"] = -returns_df["TotalLineRevenue"]
returns_df["Month"] = returns_df["InvoiceDate"].dt.to_period("M").dt.to_timestamp()

sales_df = df_products_clean[
    (df_products_clean["Quantity"] > 0) & (df_products_clean["UnitPrice"] > 0) &
    (df_products_clean["TotalLineRevenue"] > 0) & ~is_cancel_invoice & has_date
].copy()
sales_df["Month"] = sales_df["InvoiceDate"].dt.to_period("M").dt.to_timestamp()

# --- sanity checks ---
print("Return lines                                    :", f"{len(returns_df):,}")
print("Sales lines                                     :", f"{len(sales_df):,}")
print("Negative quantity outside cancellation invoices :", int(((df_products_clean['Quantity'] < 0) & ~is_cancel_invoice).sum()))
print("Positive quantity inside cancellation invoices  :", int(((df_products_clean['Quantity'] > 0) & is_cancel_invoice).sum()))
print("Return lines with zero value (UnitPrice = 0)    :", int((returns_df['Return_Value'] <= 0).sum()))

Return lines                                    : 8,704
Sales lines                                     : 528,913
Negative quantity outside cancellation invoices : 0
Positive quantity inside cancellation invoices  : 0
Return lines with zero value (UnitPrice = 0)    : 0


#### 7.9.2 How Big Are Returns?

The first table compares returns with sales on four measures: **lines**, **invoices**, **units** and **value**. The last column is returns as a percentage of sales, so you can see whether returns matter more in money than in line count (large-ticket items being returned shows up as a higher value %).

The second table splits the return value between **registered customers** and **guests** (return invoices without a `CustomerID`). Only registered customers can be linked to a customer profile later.

In [74]:
rows = [
    ("Lines",    len(returns_df),                         len(sales_df)),
    ("Invoices", returns_df["InvoiceNo"].nunique(),       sales_df["InvoiceNo"].nunique()),
    ("Units",    returns_df["Return_Units"].sum(),        sales_df["Quantity"].sum()),
    ("Value",    returns_df["Return_Value"].sum(),        sales_df["TotalLineRevenue"].sum()),
]
kpi = pd.DataFrame(rows, columns=["Measure", "Returns", "Sales"])
kpi["Returns as % of sales"] = kpi["Returns"] / kpi["Sales"] * 100
display(kpi.round(2))

who = (returns_df.assign(Who=np.where(returns_df["CustomerID"].isna(), "Guest", "Registered customer"))
       .groupby("Who")
       .agg(Return_Lines=("InvoiceNo", "size"), Return_Invoices=("InvoiceNo", "nunique"),
            Return_Value=("Return_Value", "sum")))
who["Value_%"] = who["Return_Value"] / who["Return_Value"].sum() * 100
display(who.round(1))

,Measure,Returns,Sales,Returns as % of sales
0,Lines,8704.00,528913.00,1.65
1,Invoices,3422.00,20521.00,16.68
2,Units,272030.00,5643543.00,4.82
3,Value,478724.18,10424888.31,4.59


,Return_Lines,Return_Invoices,Return_Value,Value_%
Who,,,,
Guest,165,39,4320.8,0.9
Registered customer,8539,3383,474403.3,99.1


#### 7.9.3 Size of Returns — Are There Extreme Lines?

Return values are very uneven: most lines are small, but a handful are enormous (for example a full wholesale order cancelled in one go). Such lines can dominate every total and every top-10 list.

I flag the **top 0.1% of return lines by value** as `Is_Extreme` (a data-driven cut-off, not a hand-picked number). Nothing is deleted. The flag lets the charts below offer an **"excluding extreme lines"** view.

- The table lists the 10 biggest return lines and what share of all return value the extreme lines carry.
- The histogram shows the distribution of return-line values on a **log10 scale** (otherwise only one bar would be visible).

In [75]:
extreme_cutoff = returns_df["Return_Value"].quantile(0.999)
returns_df["Is_Extreme"] = returns_df["Return_Value"] > extreme_cutoff

top_return_lines = returns_df.nlargest(10, "Return_Value")[
    ["InvoiceNo", "InvoiceDate", "StockCode", "Description", "Return_Units", "UnitPrice", "Return_Value", "CustomerID"]]
display(top_return_lines)

extreme_share = returns_df.loc[returns_df["Is_Extreme"], "Return_Value"].sum() / returns_df["Return_Value"].sum()
print(f"Extreme cut-off (99.9th percentile of line value): {extreme_cutoff:,.0f}")
print(f"Extreme lines: {int(returns_df['Is_Extreme'].sum())} "
      f"-> {extreme_share:.1%} of total return value")

hist_df = returns_df[returns_df["Return_Value"] > 0].assign(
    log10_Return_Value=lambda d: np.log10(d["Return_Value"]))
fig = px.histogram(hist_df, x="log10_Return_Value", nbins=50, marginal="box",
                   title="Distribution of return-line value (log10 scale)",
                   color_discrete_sequence=["#E67E22"])
fig.update_layout(title_x=0.5, height=430, bargap=0.05)
fig.show()

,InvoiceNo,InvoiceDate,StockCode,Description,Return_Units,UnitPrice,Return_Value,CustomerID
540422,C581484,2011-12-09 09:27:00,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.60,16446
61624,C541433,2011-01-18 10:17:00,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60,12346
160145,C550456,2011-04-18 13:08:00,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10,6539.40,15749
160143,C550456,2011-04-18 13:08:00,85123A,WHITE HANGING HEART T-LIGHT HOLDER,1930,2.55,4921.50,15749
160141,C550456,2011-04-18 13:08:00,48185,DOORMAT FAIRY CAKE,670,6.75,4522.50,15749
349750,C567527,2011-09-21 09:16:00,23113,PANTRY CHOPPING BOARD,756,5.06,3825.36,17450
160144,C550456,2011-04-18 13:08:00,21175,GIN + TONIC DIET METAL SIGN,2000,1.85,3700.00,15749
160142,C550456,2011-04-18 13:08:00,47566B,TEA TIME PARTY BUNTING,1300,2.55,3315.00,15749
320580,C565044,2011-08-31 17:02:00,22191,IVORY DINER WALL CLOCK,318,7.65,2432.70,12931
390547,C570556,2011-10-11 11:10:00,22273,FELTCRAFT DOLL MOLLY,720,2.55,1836.00,16029


Extreme cut-off (99.9th percentile of line value): 2,013
Extreme lines: 9 -> 57.4% of total return value


#### 7.9.4 Returns Over Time

Bars = return value per month (left axis). Line = return rate, i.e. return value as a % of that month's sales value (right axis).

Use the dropdown to switch between **all returns** and **excluding the extreme lines**. If a single month spikes in the first view but not in the second, the spike was caused by one or two giant cancellations and not by a general change in customer behaviour.

In [76]:
sales_m = sales_df.groupby("Month")["TotalLineRevenue"].sum().rename("Sales_Value")
ret_all_m = returns_df.groupby("Month")["Return_Value"].sum().rename("Return_Value_All")
ret_core_m = returns_df[~returns_df["Is_Extreme"]].groupby("Month")["Return_Value"].sum().rename("Return_Value_Core")

monthly_ret = pd.concat([sales_m, ret_all_m, ret_core_m], axis=1).fillna(0).reset_index()
monthly_ret = monthly_ret.rename(columns={monthly_ret.columns[0]: "Month"})
has_sales = monthly_ret["Sales_Value"] > 0          # a month with no sales has no meaningful rate
monthly_ret["Rate_All_%"] = np.where(has_sales, monthly_ret["Return_Value_All"] / monthly_ret["Sales_Value"] * 100, np.nan)
monthly_ret["Rate_Core_%"] = np.where(has_sales, monthly_ret["Return_Value_Core"] / monthly_ret["Sales_Value"] * 100, np.nan)
display(monthly_ret.round(1))

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=monthly_ret["Month"], y=monthly_ret["Return_Value_All"], name="Return value (all)",
                     marker_color="#E67E22", opacity=0.75), secondary_y=False)
fig.add_trace(go.Scatter(x=monthly_ret["Month"], y=monthly_ret["Rate_All_%"], name="Return rate % (all)",
                         mode="lines+markers", line=dict(color="#2C3E50", width=3)), secondary_y=True)
fig.add_trace(go.Bar(x=monthly_ret["Month"], y=monthly_ret["Return_Value_Core"], name="Return value (excl. extreme)",
                     marker_color="#E67E22", opacity=0.75, visible=False), secondary_y=False)
fig.add_trace(go.Scatter(x=monthly_ret["Month"], y=monthly_ret["Rate_Core_%"], name="Return rate % (excl. extreme)",
                         mode="lines+markers", line=dict(color="#2C3E50", width=3), visible=False), secondary_y=True)
fig.update_yaxes(title_text="Return value", secondary_y=False)
fig.update_yaxes(title_text="Return rate (% of monthly sales value)", secondary_y=True)
fig.update_layout(title="Monthly returns: value and rate", title_x=0.5, height=480, hovermode="x unified")
add_dropdown(fig, [
    dict(label="All returns", method="update", args=[{"visible": [True, True, False, False]}]),
    dict(label="Excluding extreme lines", method="update", args=[{"visible": [False, False, True, True]}]),
], x=0.0)
fig.show()

C:\Users\hamas\AppData\Local\Temp\ipykernel_22640\2521409962.py:10: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(monthly_ret.round(1))


,Month,Sales_Value,Return_Value_All,Return_Value_Core,Rate_All_%,Rate_Core_%
0,2010-12-01,787957.9,17607.9,17607.9,2.2,2.2
1,2011-01-01,682682.9,91526.3,14342.7,13.4,2.1
2,2011-02-01,511280.6,8375.9,8375.9,1.6,1.6
3,2011-03-01,709190.3,10675.3,10675.3,1.5,1.5
4,2011-04-01,522784.4,33316.9,10318.5,6.4,2.0
5,2011-05-01,753568.4,8948.2,8948.2,1.2,1.2
6,2011-06-01,764820.6,13760.8,13760.8,1.8,1.8
7,2011-07-01,702587.8,11409.2,11409.2,1.6,1.6
8,2011-08-01,742826.4,22929.5,20496.8,3.1,2.8
9,2011-09-01,1039808.0,17044.7,13219.3,1.6,1.3


#### 7.9.5 Which Products Are Returned Most?

Top 15 products on three different measures. The dropdown has six views (3 measures × with / without the extreme lines):

- **Return value** — where the money goes. Dominated by expensive items and by the extreme lines.
- **Return units** — how many pieces come back.
- **Return rate (% of the product's sales value)** — which products are returned *disproportionately* to how much they sell. To keep this list meaningful, only products with at least `MIN_UNITS_SOLD` units sold **and** `MIN_RETURN_LINES` return lines are ranked; otherwise a product sold twice and returned once would top the list.

The first view is a good place to find products worth a quality check; the rate view is the one to use when asking "what is wrong with this product?".

In [77]:
MIN_UNITS_SOLD = 100       # a product needs at least this many units sold to be ranked by rate
MIN_RETURN_LINES = 3       # ... and at least this many return lines

description_by_code = df_products_clean.groupby("StockCode")["Description"].first()
product_sales = sales_df.groupby("StockCode").agg(Sales_Value=("TotalLineRevenue", "sum"),
                                                  Sales_Units=("Quantity", "sum"))

def product_returns(rdf):
    r = rdf.groupby("StockCode").agg(Return_Value=("Return_Value", "sum"),
                                     Return_Units=("Return_Units", "sum"),
                                     Return_Lines=("InvoiceNo", "size"))
    out = r.join(product_sales, how="left").fillna({"Sales_Value": 0, "Sales_Units": 0}).reset_index()
    out["Return_Rate_%"] = np.where(out["Sales_Value"] > 0, out["Return_Value"] / out["Sales_Value"] * 100, np.nan)
    out["Product_Label"] = (out["StockCode"].astype(str) + " - " +
                            out["StockCode"].map(description_by_code).fillna("No description").astype(str).str.slice(0, 35))
    return out

prod_all = product_returns(returns_df)
prod_core = product_returns(returns_df[~returns_df["Is_Extreme"]])

def top_products(p, metric, n=15):
    d = p
    if metric == "Return_Rate_%":
        d = p[(p["Sales_Units"] >= MIN_UNITS_SOLD) & (p["Return_Lines"] >= MIN_RETURN_LINES)]
    return d.nlargest(n, metric).sort_values(metric)

metric_specs = [("Return_Value", "Return value"), ("Return_Units", "Return units"),
                ("Return_Rate_%", "Return rate (% of sales value)")]
view_specs = []                      # (button label, metric, data)
for metric, metric_label in metric_specs:
    view_specs.append((f"{metric_label} - all returns", metric, prod_all))
    view_specs.append((f"{metric_label} - excl. extreme lines", metric, prod_core))

fig = go.Figure()
for k, (label, metric, data) in enumerate(view_specs):
    d = top_products(data, metric)
    fig.add_trace(go.Bar(x=d[metric], y=d["Product_Label"], orientation="h", name=label,
                         marker_color="#E67E22", visible=(k == 0), showlegend=False,
                         hovertemplate="%{y}<br>%{x:,.1f}<extra></extra>"))
fig.update_yaxes(type="category")
fig.update_layout(title="Top 15 returned products", title_x=0.5, height=560, xaxis_title="")
add_dropdown(fig, [dict(label=label, method="update",
                        args=[{"visible": [j == k for j in range(len(view_specs))]},
                              {"title.text": f"Top 15 returned products: {label}"}])
                   for k, (label, _, _) in enumerate(view_specs)], x=0.0)
fig.show()

#### 7.9.6 Returns by Country

Countries with very little business can show absurd return rates, so only countries with at least `MIN_COUNTRY_SALES` of sales value are included. Dropdown views:

- **Return rate (%)** — which markets return the most relative to what they buy.
- **Return value** — where the returned money is, which is mostly a question of market size.
- **Return value, hiding the United Kingdom** — the UK is about 85% of sales, so it flattens every other bar; hiding it makes the international markets comparable.

In [78]:
MIN_COUNTRY_SALES = 5000

country_sales = sales_df.groupby("Country")["TotalLineRevenue"].sum().rename("Sales_Value")
country_returns = returns_df.groupby("Country")["Return_Value"].sum().rename("Return_Value")
country_ret = pd.concat([country_sales, country_returns], axis=1).fillna(0).reset_index()
country_ret = country_ret.rename(columns={country_ret.columns[0]: "Country"})
country_ret = country_ret[(country_ret["Sales_Value"] >= MIN_COUNTRY_SALES) & (country_ret["Return_Value"] > 0)].copy()
country_ret["Return_Rate_%"] = country_ret["Return_Value"] / country_ret["Sales_Value"] * 100
display(country_ret.sort_values("Return_Value", ascending=False).round(1).head(15))

def country_view(df, metric, n=15):
    d = df.nlargest(n, metric).sort_values(metric)
    return d[metric].round(1).tolist(), d["Country"].tolist()

views = {
    "Return rate (%)":                          (country_view(country_ret, "Return_Rate_%"), "Return rate (% of sales value)"),
    "Return value":                             (country_view(country_ret, "Return_Value"), "Return value"),
    "Return value - hide United Kingdom":       (country_view(country_ret[country_ret["Country"] != "United Kingdom"], "Return_Value"), "Return value"),
}
(first_x, first_y), first_title = views["Return rate (%)"]

fig = go.Figure(go.Bar(x=first_x, y=first_y, orientation="h", marker_color="#E67E22"))
fig.update_yaxes(type="category")
fig.update_layout(title="Returns by country", title_x=0.5, height=520, xaxis_title=first_title)
add_dropdown(fig, [dict(label=name, method="update",
                        args=[{"x": [v[0][0]], "y": [v[0][1]]}, {"xaxis.title.text": v[1]}])
                   for name, v in views.items()], x=0.0)
fig.show()

,Country,Sales_Value,Return_Value,Return_Rate_%
36,United Kingdom,8898430.1,445843.4,5.0
10,EIRE,271982.6,11500.8,4.2
14,Germany,205577.8,4761.5,2.3
31,Spain,55771.9,3959.9,7.1
13,France,184582.7,2419.4,1.3
20,Japan,37416.4,1996.6,5.3
32,Sweden,36839.3,1662.4,4.5
0,Australia,139120.5,1181.3,0.8
33,Switzerland,53096.4,582.6,1.1
19,Italy,15820.2,543.9,3.4


#### 7.9.7 Which Customers Return the Most?

A per-customer table, **`customer_returns`**, is built for registered customers who have at least one return line:

| Column | Meaning |
|---|---|
| `Return_Invoices`, `Return_Units`, `Return_Value` | Their returns. |
| `Sales_Value`, `Sales_Invoices` | Their valid purchases (the same base as the RFM). |
| `Return_Rate_%` | `Return_Value ÷ Sales_Value × 100`. |
| `Net_Value` | `Sales_Value − Return_Value` — the *net* Monetary value. |
| `Return_Bucket` | The return rate grouped into `<5%`, `5–20%`, `20–50%`, `>50%` (or `No purchases` for customers who returned but never bought in the valid sales data). |

The printed lines answer three questions: how many customers return anything at all, how many returned more than they bought, and how many only appear in the returns.

**Chart.** Each dot is a customer: purchases on the x-axis, returns on the y-axis (log–log). The dashed diagonal is *returns = purchases*: dots **above** the line returned more than they bought. Colour = return-rate bucket. Use the **bucket filter** to isolate, for example, only the customers whose return rate is above 50%, and the **scale** menu to switch to linear axes.

In [79]:
registered_returns = returns_df.dropna(subset=["CustomerID"])
cust_ret = registered_returns.groupby("CustomerID").agg(
    Return_Invoices=("InvoiceNo", "nunique"),
    Return_Units=("Return_Units", "sum"),
    Return_Value=("Return_Value", "sum"))
cust_sales = sales_df.dropna(subset=["CustomerID"]).groupby("CustomerID").agg(
    Sales_Value=("TotalLineRevenue", "sum"),
    Sales_Invoices=("InvoiceNo", "nunique"))

customer_returns = cust_ret.join(cust_sales, how="left").fillna({"Sales_Value": 0, "Sales_Invoices": 0}).reset_index()
customer_returns["Return_Rate_%"] = np.where(customer_returns["Sales_Value"] > 0,
                                             customer_returns["Return_Value"] / customer_returns["Sales_Value"] * 100,
                                             np.nan)
customer_returns["Net_Value"] = customer_returns["Sales_Value"] - customer_returns["Return_Value"]

BUCKET_ORDER = ["<5%", "5-20%", "20-50%", ">50%", "No purchases"]
customer_returns["Return_Bucket"] = (
    pd.cut(customer_returns["Return_Rate_%"], bins=[0, 5, 20, 50, np.inf], labels=BUCKET_ORDER[:4], right=False)
    .astype(object).fillna("No purchases")
)

n_buyers = sales_df["CustomerID"].nunique()
print(f"Registered customers with valid purchases : {n_buyers:,}")
print(f"Customers with at least one return        : {len(customer_returns):,} "
      f"({(customer_returns['Sales_Value'] > 0).sum() / n_buyers:.1%} of the buyers)")
print(f"Returned more than they bought            : {int((customer_returns['Net_Value'] < 0).sum()):,}")
print(f"Only appear in returns (no valid purchase): {int((customer_returns['Sales_Value'] == 0).sum()):,}")
display(customer_returns["Return_Bucket"].value_counts().reindex(BUCKET_ORDER).fillna(0).astype(int).to_frame("Customers"))
display(customer_returns[["Return_Invoices", "Return_Units", "Return_Value", "Sales_Value", "Return_Rate_%"]].describe().T.round(1))

# ---- scatter: purchases vs returns, with bucket filter and scale switch ----
plot_df = customer_returns[(customer_returns["Sales_Value"] > 0) & (customer_returns["Return_Value"] > 0)]
fig = px.scatter(plot_df, x="Sales_Value", y="Return_Value", color="Return_Bucket",
                 category_orders={"Return_Bucket": BUCKET_ORDER[:4]},
                 color_discrete_sequence=["#95A5A6", "#F5B041", "#E67E22", "#C0392B"],
                 hover_data=["CustomerID", "Return_Invoices", "Return_Rate_%"],
                 log_x=True, log_y=True, opacity=0.7,
                 title="Customer purchases vs returns (above the dashed line = returned more than bought)")
lo = float(min(plot_df["Sales_Value"].min(), plot_df["Return_Value"].min()))
hi = float(max(plot_df["Sales_Value"].max(), plot_df["Return_Value"].max()))
fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines", name="Returns = purchases",
                         line=dict(color="#2C3E50", dash="dash")))
fig.update_layout(title_x=0.5, height=560)
add_dropdown(fig, visibility_buttons(fig, BUCKET_ORDER[:4], always=("Returns = purchases",)), x=0.0)
add_dropdown(fig, [
    dict(label="Log-log (default)", method="relayout",
         args=[{"xaxis.type": "log", "yaxis.type": "log", "xaxis.autorange": True, "yaxis.autorange": True}]),
    dict(label="Linear", method="relayout",
         args=[{"xaxis.type": "linear", "yaxis.type": "linear", "xaxis.autorange": True, "yaxis.autorange": True}]),
], x=0.2)
fig.show()

Registered customers with valid purchases : 4,335
Customers with at least one return        : 1,534 (34.7% of the buyers)
Returned more than they bought            : 30
Only appear in returns (no valid purchase): 28


,Customers
Return_Bucket,
<5%,1130
5-20%,280
20-50%,70
>50%,26
No purchases,28


,count,mean,std,min,25%,50%,75%,max
Return_Invoices,1534.0,2.2,2.8,1.0,1.0,1.0,2.0,45.0
Return_Units,1534.0,175.7,2838.3,1.0,3.0,9.0,26.0,80995.0
Return_Value,1534.0,309.3,4798.5,0.4,13.0,30.0,75.2,168469.6
Sales_Value,1534.0,3971.6,13052.8,0.0,678.7,1550.2,3231.0,279860.9
Return_Rate_%,1506.0,6.1,14.3,0.0,0.9,2.0,5.0,237.0


#### 7.9.8 Are Returns Concentrated in a Few Customers?

The customers who returned something are sorted from the biggest returner to the smallest. The curve shows how much of the total return value the top X% of *returning* customers account for; the dotted diagonal is the "returns spread evenly" case. If the curve is far above the diagonal, a small group of customers drives most of the return value, which means returns are a **customer-management** question (a few accounts to talk to) rather than a company-wide problem.

In [80]:
cr_sorted = customer_returns.sort_values("Return_Value", ascending=False).reset_index(drop=True)
cr_sorted["Cum_Customer_Share"] = np.arange(1, len(cr_sorted) + 1) / len(cr_sorted)
cr_sorted["Cum_Return_Share"] = cr_sorted["Return_Value"].cumsum() / cr_sorted["Return_Value"].sum()

for pct in (0.01, 0.05, 0.10, 0.20):
    share = cr_sorted.loc[cr_sorted["Cum_Customer_Share"] <= pct, "Cum_Return_Share"]
    print(f"Top {pct:>4.0%} of returning customers -> {share.iloc[-1] if len(share) else 0:.1%} of return value")

fig = go.Figure()
fig.add_trace(go.Scatter(x=cr_sorted["Cum_Customer_Share"], y=cr_sorted["Cum_Return_Share"], mode="lines",
                         fill="tozeroy", name="Cumulative return value", line=dict(color="#E67E22", width=3)))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Perfect equality",
                         line=dict(color="#95A5A6", dash="dot")))
fig.update_layout(title="Concentration of return value among returning customers", title_x=0.5,
                  xaxis=dict(title="Cumulative share of returning customers (biggest first)", tickformat=".0%"),
                  yaxis=dict(title="Cumulative share of return value", tickformat=".0%"),
                  hovermode="x unified", height=450)
fig.show()

Top   1% of returning customers -> 70.8% of return value
Top   5% of returning customers -> 82.4% of return value
Top  10% of returning customers -> 87.7% of return value
Top  20% of returning customers -> 92.5% of return value


#### 7.9.9 How Long After the Purchase Do Returns Happen?

Since a return has no link to its original invoice, I **match each return line to the closest earlier purchase of the same product by the same customer** (`pd.merge_asof`, backwards in time). This only works for registered customers.

- **Match status** shows how many return lines could be matched. Lines **without an earlier purchase in the data** are returns of items bought before 01/12/2010, bought under another ID, or bought as a guest, so they cannot be explained by this dataset.
- **Days between purchase and return** is computed for the matched lines only. The histogram shows its distribution. Use the dropdown to zoom to the first 30 or 90 days, where most of the action usually is.

In [81]:
sold_reg = (sales_df.dropna(subset=["CustomerID"])[["CustomerID", "StockCode", "InvoiceDate"]]
            .rename(columns={"InvoiceDate": "Purchase_Date"}))
sold_reg["CustomerID"] = sold_reg["CustomerID"].astype("int64")
sold_reg = sold_reg.sort_values("Purchase_Date")

ret_reg = registered_returns[["InvoiceNo", "CustomerID", "StockCode", "InvoiceDate", "Return_Units", "Return_Value"]].copy()
ret_reg["CustomerID"] = ret_reg["CustomerID"].astype("int64")
ret_reg = ret_reg.sort_values("InvoiceDate")

matched = pd.merge_asof(ret_reg, sold_reg, left_on="InvoiceDate", right_on="Purchase_Date",
                        by=["CustomerID", "StockCode"], direction="backward")
matched["Days_To_Return"] = (matched["InvoiceDate"] - matched["Purchase_Date"]).dt.days
matched["Match_Status"] = np.where(matched["Purchase_Date"].isna(),
                                   "No earlier purchase in the data", "Matched to an earlier purchase")

status = (matched.groupby("Match_Status")
          .agg(Return_Lines=("InvoiceNo", "size"), Return_Value=("Return_Value", "sum")).reset_index())
status["Lines_%"] = status["Return_Lines"] / status["Return_Lines"].sum() * 100
status["Value_%"] = status["Return_Value"] / status["Return_Value"].sum() * 100
display(status.round(1))

days = matched.loc[matched["Days_To_Return"].notna(), "Days_To_Return"]
if len(days):
    print(f"Days between purchase and return -> median: {days.median():.0f} | 75th pct: {days.quantile(.75):.0f} | 90th pct: {days.quantile(.90):.0f}")
    fig = px.histogram(matched[matched["Days_To_Return"].notna()], x="Days_To_Return", nbins=60, marginal="box",
                       title="Days between the purchase and the return (matched lines only)",
                       color_discrete_sequence=["#E67E22"])
    fig.update_layout(title_x=0.5, height=450, bargap=0.05, xaxis_title="Days between purchase and return")
    add_dropdown(fig, [
        dict(label="Full range", method="relayout", args=[{"xaxis.autorange": True}]),
        dict(label="First 30 days", method="relayout", args=[{"xaxis.range": [0, 30]}]),
        dict(label="First 90 days", method="relayout", args=[{"xaxis.range": [0, 90]}]),
    ], x=0.0)
    fig.show()

,Match_Status,Return_Lines,Return_Value,Lines_%,Value_%
0,Matched to an earlier purchase,7508,452203.3,87.9,95.3
1,No earlier purchase in the data,1031,22200.0,12.1,4.7


Days between purchase and return -> median: 9 | 75th pct: 24 | 90th pct: 66


#### 7.9.10 Key Numbers

The cell below prints the headline numbers of the returns analysis so that the conclusions rest on real output: overall return rate, the concentration of returns, the worst product and the customers that returned more than they bought.

In [82]:
val_rate = returns_df["Return_Value"].sum() / sales_df["TotalLineRevenue"].sum()
line_rate = len(returns_df) / len(sales_df)
print(f"Overall return rate by value : {val_rate:.2%} of sales value")
print(f"Overall return rate by lines : {line_rate:.2%} of sales lines")
print(f"Extreme lines (top 0.1%)     : {extreme_share:.1%} of return value")

worst_month = monthly_ret.loc[monthly_ret["Rate_All_%"].idxmax()]
print(f"Highest monthly return rate  : {worst_month['Month']:%B %Y} -> {worst_month['Rate_All_%']:.1f}% (excl. extreme lines: {worst_month['Rate_Core_%']:.1f}%)")

top_prod = prod_all.nlargest(1, "Return_Value").iloc[0]
print(f"Biggest returned product     : {top_prod['Product_Label']} -> {top_prod['Return_Value']:,.0f}")

top10_share = cr_sorted.loc[cr_sorted["Cum_Customer_Share"] <= 0.10, "Cum_Return_Share"]
print(f"Top 10% of returning customers hold {top10_share.iloc[-1] if len(top10_share) else 0:.1%} of return value")
neg = customer_returns[customer_returns["Net_Value"] < 0]
print(f"Customers with negative net value: {len(neg):,} (net value {neg['Net_Value'].sum():,.0f})")

Overall return rate by value : 4.59% of sales value
Overall return rate by lines : 1.65% of sales lines
Extreme lines (top 0.1%)     : 57.4% of return value
Highest monthly return rate  : December 2011 -> 28.2% (excl. extreme lines: 0.9%)
Biggest returned product     : 23843 - PAPER CRAFT , LITTLE BIRDIE -> 168,470
Top 10% of returning customers hold 87.7% of return value
Customers with negative net value: 30 (net value -4,160)


**How this connects to the RFM.** The customer RFM in section 8 uses sales only, so `Monetary` is a **gross** figure. `customer_returns` (built in 7.9.7) already contains `Return_Value` and `Net_Value` per customer, so the natural next step is to merge it into `customer_rfm`, compute a **net Monetary**, and check how many customers change segment compared with the gross version.

### 7.10 EDA Findings — Summary Before Moving to RFM

The exact numbers are in the cell above; this is what I took away:

- **Time:** the month with the highest revenue is printed above. Since the data covers about one year (12 months), I cannot tell whether this is a seasonal pattern that repeats or a one-off event.
- **Hours and days:** the number of invoices is concentrated between 10:00 and 15:00, and the busiest hour is 12. The days with no sales are printed above.
- **Basket behaviour:** the mean invoice value (**508.01**) is clearly higher than the median (**298.44**), so the distribution of invoice values is **right-skewed**: a few large invoices (probably wholesale) pull the mean up. That is why I prefer the median and the log scale in the analysis.
- **Pareto:** only **830 products (21.2%)** generate 80% of the revenue, so the 80/20 rule holds quite closely here.
- **Products:** the cell above prints how many products are shared between the top 10 by revenue and the top 10 by quantity.
- **Returns:** analysed in detail in 7.9 (value, months, products, countries, customers, and the time between purchase and return).
- **Customers:** there are 4,335 registered customers in the sales, and I look at the concentration of revenue among them in detail in Section 8.

> These conclusions prepare the ground for the RFM: if revenue is concentrated in a few customers and products, segmentation will make a difference to decisions.

## 8. Customer-Level RFM Analysis

**RFM** summarises the behaviour of every customer with just three numbers:

| Metric | Question it answers | Which direction is good? |
|---|---|---|
| **Recency (R)** | How many days have passed since the customer's last purchase? | Lower is better |
| **Frequency (F)** | How many separate purchases did the customer make? We count **unique invoices**, not lines, because a single invoice can contain dozens of lines. | Higher is better |
| **Monetary (M)** | How much money did the customer spend in total? | Higher is better |

**Plan for this section**

1. Build R, F and M for every customer (8.1 – 8.3) and look at their distributions (8.4).
2. Convert each metric into a 1–5 score (8.5) and check that the scores behave sensibly (8.6).
3. Combine the scores into **4 main customer segments: Champions, Loyal, At Risk and Lost**. Each main segment is made of finer **sub-segments** (11 in total) so we keep the detail without drowning in it (8.7).
4. Visualise the segments with interactive charts. Every chart has **filters** (dropdown menus) so that the huge segments/outliers do not hide the smaller ones (8.8).
5. Check that the result is robust (8.9) and turn each segment into a business action (8.10).

### 8.1 Valid Customer Transactions

RFM for customers is computed only on rows that satisfy **all** of these conditions:

- `Quantity > 0` and `UnitPrice > 0` (real sales, no returns or free items),
- the invoice is not a cancellation (invoice numbers starting with `C`),
- the row has a **CustomerID** (guest checkouts have no identity, so they cannot be profiled).

The funnel below shows how many rows are left after each filter, from the raw file down to the rows that RFM is computed on.

In [83]:
# Prepare valid customer purchase transactions

customer_transactions = df_products_clean[
    (df_products_clean["Quantity"] > 0) &
    (df_products_clean["UnitPrice"] > 0) &
    (df_products_clean["TotalLineRevenue"] > 0) &
    (df_products_clean["CustomerID"].notna()) &
    (df_products_clean["InvoiceDate"].notna()) &
    (~df_products_clean["InvoiceNo"].astype(str).str.startswith("C"))
].copy()

print("Customer Transactions:", len(customer_transactions))
print("Unique Customers:", customer_transactions["CustomerID"].nunique())
print("Unique Invoices:", customer_transactions["InvoiceNo"].nunique())

Customer Transactions: 396374
Unique Customers: 4335
Unique Invoices: 18405


In [84]:
funnel_steps = pd.DataFrame({
    "Step": [
        "1. Raw data",
        "2. Real products only",
        "3. After dropping unrecoverable zero-price lines",
        "4. Sales only (Quantity > 0)",
        "5. Registered customers only (RFM input)",
    ],
    "Rows": [
        len(df),
        int((df["ItemType"] == "Product").sum()),
        len(df_products_clean),
        int((df_products_clean["Quantity"] > 0).sum()),
        len(customer_transactions),
    ],
})
funnel_steps["% of raw"] = (funnel_steps["Rows"] / funnel_steps["Rows"].iloc[0] * 100).round(1)
display(funnel_steps)

fig = go.Figure(go.Funnel(
    y=funnel_steps["Step"], x=funnel_steps["Rows"],
    textinfo="value+percent initial",
    marker=dict(color=["#2E86DE", "#3F9BE8", "#5BAEF0", "#F5B041", "#1B7F4C"]),
))
fig.update_layout(title="From raw rows to the rows used in Customer RFM", title_x=0.5,
                  height=430, margin=dict(l=320))
fig.show()

,Step,Rows,% of raw
0,1. Raw data,541909,100.0
1,2. Real products only,537636,99.2
2,3. After dropping unrecoverable zero-price lines,537617,99.2
3,4. Sales only (Quantity > 0),528913,97.6
4,5. Registered customers only (RFM input),396374,73.1


### 8.2 Snapshot Date

Recency needs a reference date ("today"). I use **the last date in the data + 1 day** (10/12/2011). The extra day makes a customer who bought on the very last day get Recency = 1 instead of 0, which keeps the metric strictly positive.

In [85]:
snapshot_date = (
    customer_transactions["InvoiceDate"].max()
    + pd.Timedelta(days=1)
)

print("Snapshot Date:", snapshot_date)

Snapshot Date: 2011-12-10 12:50:00


### 8.3 Building R, F, M per Customer

One row per customer, built with `groupby("CustomerID")`:

- **Recency** — the customer's most recent purchase date (converted to "days before the snapshot date" right after the aggregation),
- **Frequency** — the number of unique invoices (`nunique` on `InvoiceNo`),
- **Monetary** — the sum of `TotalLineRevenue` over all of the customer's lines.

In [86]:
customer_rfm = (
    customer_transactions
    .groupby("CustomerID")
    .agg(
        Recency=("InvoiceDate", "max"),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("TotalLineRevenue", "sum")
    )
    .reset_index()
)

# Calculate Recency in days
customer_rfm["Recency"] = (
    snapshot_date - customer_rfm["Recency"]
).dt.days

customer_rfm = customer_rfm[
    ["CustomerID", "Recency", "Frequency", "Monetary"]
]

display(customer_rfm.head(10))

,CustomerID,Recency,Frequency,Monetary
0,12346,326,1,77183.60
1,12347,2,7,4310.00
2,12348,75,4,1437.24
3,12349,19,1,1457.55
4,12350,310,1,294.40
5,12352,36,7,1385.74
6,12353,204,1,89.00
7,12354,232,1,1079.40
8,12355,214,1,459.40
9,12356,23,3,2487.43


In [87]:
display(
    customer_rfm[
        ["Recency", "Frequency", "Monetary"]
    ].describe().T
)

,count,mean,std,min,25%,50%,75%,max
Recency,4335.0,92.684890,100.172622,1.00,18.00,51.00,143.000,374.0
Frequency,4335.0,4.245675,7.636337,1.00,1.00,2.00,5.000,206.0
Monetary,4335.0,2022.250799,8917.297952,3.75,305.58,668.14,1632.775,279860.9


### 8.4 Distributions

What `describe()` above tells us:

- **Monetary:** the mean (≈ 2,022) is far above the median (≈ 668) and the maximum is ≈ 279,861 — a few customers spend enormous amounts.
- **Frequency:** the median customer made only 2 purchases, yet one customer made 206.
- **Recency:** the median is 51 days and the mean ≈ 93 days.

So **Frequency and Monetary are heavily right-skewed** and contain big outliers (most likely wholesale buyers). Two consequences for the rest of the analysis:

1. The histograms below use a **log10 scale** for F and M; otherwise one tall bar and a long empty tail is all you would see.
2. The scoring step (8.5) works on **ranks (quintiles)** instead of raw values, so a few huge customers cannot stretch the scale and squash everybody else into one score.

In [88]:
# histogram + box plot, log10 for the skewed metrics
rfm_dist_specs = [
    ("Recency",   "Recency: days since last purchase",           False),
    ("Frequency", "Frequency: unique invoices (log10 scale)",    True),
    ("Monetary",  "Monetary: total spend (log10 scale)",         True),
]

for col, title, use_log in rfm_dist_specs:
    plot_df = customer_rfm[["CustomerID", col]].copy()
    x_col = col
    if use_log:
        x_col = f"log10_{col}"
        plot_df[x_col] = np.log10(plot_df[col])

    fig = px.histogram(plot_df, x=x_col, nbins=50, marginal="box",
                       title=title, color_discrete_sequence=["#2E86DE"])
    fig.update_layout(title_x=0.5, height=420, bargap=0.05)
    fig.show()

### 8.5 RFM Scoring (1–5)

Each metric is split into 5 equal-sized groups (**quintiles**, via `pd.qcut`) and every customer gets a score from 1 to 5 per metric:

- **R_Score** is *reversed*: the most recent 20% of customers get **5**, because a lower Recency is better.
- **F_Score** and **M_Score**: the top 20% get **5**.

Combining the three scores gives the 3-character `RFM_Score` (for example `555` = best on everything, `111` = worst on everything).

**Why `rank(method="first")` before `qcut`?** Many customers share exactly the same value (especially Frequency = 1). With so many ties, `qcut` would produce duplicated bin edges and fail. Ranking first gives every customer a unique position. The trade-off is that customers with identical values can land in two neighbouring scores depending on row order — a small bias that is listed in the Limitations (section 10).

In [89]:
# Rank values first to handle duplicated quantile boundaries

customer_rfm["R_Score"] = pd.qcut(
    customer_rfm["Recency"].rank(method="first"),
    q=5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

customer_rfm["F_Score"] = pd.qcut(
    customer_rfm["Frequency"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

customer_rfm["M_Score"] = pd.qcut(
    customer_rfm["Monetary"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Combine scores
customer_rfm["RFM_Score"] = (
    customer_rfm["R_Score"].astype(str) +
    customer_rfm["F_Score"].astype(str) +
    customer_rfm["M_Score"].astype(str)
)

display(customer_rfm.head(10))

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,12346,326,1,77183.60,1,1,5,115
1,12347,2,7,4310.00,5,5,5,555
2,12348,75,4,1437.24,2,4,4,244
3,12349,19,1,1457.55,4,1,4,414
4,12350,310,1,294.40,1,1,2,112
5,12352,36,7,1385.74,3,5,4,354
6,12353,204,1,89.00,1,1,1,111
7,12354,232,1,1079.40,1,1,4,114
8,12355,214,1,459.40,1,1,2,112
9,12356,23,3,2487.43,4,3,5,435


### 8.6 Do the Scores Make Sense?

Before building segments on top of the scores, I check that they really represent the raw values:

- median **Recency** should *fall* as `R_Score` rises (higher score = more recent),
- median **Frequency** should *rise* with `F_Score`,
- median **Monetary** should *rise* with `M_Score`.

If any of these were not monotonic, the scoring would be broken.

In [90]:
score_check = pd.DataFrame({
    "R": customer_rfm.groupby("R_Score")["Recency"].median(),
    "F": customer_rfm.groupby("F_Score")["Frequency"].median(),
    "M": customer_rfm.groupby("M_Score")["Monetary"].median(),
}).reindex(range(1, 6))
display(score_check.round(1))

fig = make_subplots(rows=1, cols=3, subplot_titles=(
    "Median Recency (days) by R_Score", "Median Frequency by F_Score", "Median Monetary by M_Score"))
for i, (col, color) in enumerate(zip(["R", "F", "M"], ["#E67E22", "#2E86DE", "#1B7F4C"]), start=1):
    fig.add_trace(go.Bar(x=score_check.index, y=score_check[col], marker_color=color,
                         text=score_check[col].round(1), textposition="outside",
                         showlegend=False), row=1, col=i)
fig.update_xaxes(title_text="Score", dtick=1)
fig.update_layout(title_text="Do the scores behave as expected?", title_x=0.5, height=400)
fig.show()

,R,F,M
1,263.0,1.0,155.2
2,109.0,1.0,348.2
3,51.0,2.0,668.1
4,22.0,4.0,1345.6
5,5.0,9.0,3590.7


**Where do customers sit on the R × F grid?** The heatmap counts customers per (R_Score, F_Score) pair. The bar chart lists the most common of the 125 possible `RFM_Score` combinations. With 125 combinations it is hard to act on individual codes, which is exactly why section 8.7 groups them into segments.

In [91]:
# R x F: where do customers sit?
rf_counts = pd.crosstab(customer_rfm["R_Score"], customer_rfm["F_Score"])
fig = px.imshow(
    rf_counts, text_auto=True, aspect="auto", origin="lower",
    color_continuous_scale="Blues",
    labels=dict(x="F_Score (Frequency)", y="R_Score (Recency)", color="Customers"),
    title="Customers by Recency score and Frequency score",
)
fig.update_layout(title_x=0.5, height=420)
fig.show()

# The most common RFM combinations
top_combos = customer_rfm["RFM_Score"].value_counts().head(15).sort_values().reset_index()
top_combos.columns = ["RFM_Score", "Customers"]
fig = px.bar(top_combos, x="Customers", y="RFM_Score", orientation="h", text="Customers",
             title="Top 15 most common RFM combinations (out of 125 possible)",
             color_discrete_sequence=["#2E86DE"])
fig.update_yaxes(type="category")
fig.update_layout(title_x=0.5, height=480)
fig.show()

**Frequency vs Monetary (log–log).** Each dot is a customer, coloured by its Recency score. Both axes are logarithmic because a few customers are orders of magnitude bigger than the rest. Hover over a dot to see its CustomerID and RFM code.

In [92]:
# Frequency vs Monetary on log-log axes (linear axes are crushed by a few huge customers)
scatter_df = customer_rfm.assign(R_Score=customer_rfm["R_Score"].astype(str))
fig = px.scatter(
    scatter_df, x="Frequency", y="Monetary", color="R_Score",
    category_orders={"R_Score": ["1", "2", "3", "4", "5"]},
    color_discrete_sequence=px.colors.sequential.Viridis[1::2],
    hover_data=["CustomerID", "Recency", "RFM_Score"],
    log_x=True, log_y=True, opacity=0.65,
    title="Customer value: Frequency vs Monetary (colored by Recency score)",
)
fig.update_layout(title_x=0.5, height=520)
fig.show()

### 8.7 Customer Segments — 4 Main Segments, 11 Sub-Segments

A raw `RFM_Score` has 125 combinations, which is too many to interpret. I convert the scores into segments in two layers.

**Step 1 — merge F and M.** Frequency and Monetary are strongly correlated (customers who buy often usually spend more), so I merge them into one score:

`FM_Score = ceil((F_Score + M_Score) / 2)`

**Step 2 — classic RFM grid (11 sub-segments).** Each (R, FM) pair maps to a sub-segment:

| R \ FM | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| **5** | New Customers | Potential Loyalists | Potential Loyalists | Champions | Champions |
| **4** | Promising | Potential Loyalists | Potential Loyalists | Loyal Customers | Loyal Customers |
| **3** | About to Sleep | About to Sleep | Need Attention | Loyal Customers | Loyal Customers |
| **2** | Hibernating | Hibernating | At Risk | At Risk | Can't Lose Them |
| **1** | Lost | Hibernating | At Risk | At Risk | Can't Lose Them |

**Step 3 — roll up into 4 main segments.** Eleven groups are great for targeting but too many for a headline, so they are rolled up into four:

| Main segment | Sub-segments inside it | In plain words |
|---|---|---|
| **Champions** | Champions | Bought recently, buy often and spend the most. |
| **Loyal** | Loyal Customers, Potential Loyalists, New Customers, Promising | Active customers (recent, or high-value with a steady rhythm) who are not Champions yet. |
| **At Risk** | Need Attention, About to Sleep, At Risk, Can't Lose Them | Used to buy, but the gap since their last purchase is growing. *Can't Lose Them* is the high-value VIP slice of this group. |
| **Lost** | Hibernating, Lost | Haven't bought for a long time and had low value. |

The mapping lives in a single dictionary (`SEGMENT_MAP`) in the next cell, so if the business prefers a different grouping, only that dictionary needs to change. The same is true for the grid itself (`segment_grid`). Every column added here is used downstream:

- `Segment` — the main segment (4 values),
- `Segment_Detailed` — the sub-segment (11 values).

The two tables printed by the cell give, per segment: number of customers, average R / F / M, total revenue, the share of customers and of revenue, and the average order value (revenue ÷ invoices).

In [93]:
# ---------------------------------------------------------------
# FM score -> 11 detailed sub-segments -> 4 main segments
# ---------------------------------------------------------------
customer_rfm["FM_Score"] = np.ceil((customer_rfm["F_Score"] + customer_rfm["M_Score"]) / 2).astype(int)

# Classic RFM grid. Rows: R score (5 -> 1). Columns: FM score 1..5
segment_grid = {
    5: ["New Customers", "Potential Loyalists", "Potential Loyalists", "Champions", "Champions"],
    4: ["Promising", "Potential Loyalists", "Potential Loyalists", "Loyal Customers", "Loyal Customers"],
    3: ["About to Sleep", "About to Sleep", "Need Attention", "Loyal Customers", "Loyal Customers"],
    2: ["Hibernating", "Hibernating", "At Risk", "At Risk", "Can't Lose Them"],
    1: ["Lost", "Hibernating", "At Risk", "At Risk", "Can't Lose Them"],
}

# How the 11 sub-segments roll up into the 4 main segments (edit here to regroup)
SEGMENT_MAP = {
    "Champions": "Champions",
    "Loyal Customers": "Loyal", "Potential Loyalists": "Loyal", "New Customers": "Loyal", "Promising": "Loyal",
    "Need Attention": "At Risk", "About to Sleep": "At Risk", "At Risk": "At Risk", "Can't Lose Them": "At Risk",
    "Hibernating": "Lost", "Lost": "Lost",
}

SEGMENT_ORDER = ["Champions", "Loyal", "At Risk", "Lost"]
SEGMENT_COLORS = {"Champions": "#1B7F4C", "Loyal": "#2E86DE", "At Risk": "#E67E22", "Lost": "#7F8C8D"}

DETAILED_ORDER = ["Champions", "Loyal Customers", "Potential Loyalists", "New Customers", "Promising",
                  "Need Attention", "About to Sleep", "At Risk", "Can't Lose Them", "Hibernating", "Lost"]
DETAILED_COLORS = {
    "Champions": "#1B7F4C", "Loyal Customers": "#4CAF7A", "Potential Loyalists": "#9BCB6B",
    "New Customers": "#2E86DE", "Promising": "#74B9FF", "Need Attention": "#F5B041",
    "About to Sleep": "#F8C471", "At Risk": "#E67E22", "Can't Lose Them": "#C0392B",
    "Hibernating": "#95A5A6", "Lost": "#566573",
}

def assign_detailed_segment(r_score, fm_score):
    return segment_grid[int(r_score)][int(fm_score) - 1]

def assign_segment(r_score, fm_score):
    """Main segment (4 values). Used here and in the sensitivity check (8.9)."""
    return SEGMENT_MAP[assign_detailed_segment(r_score, fm_score)]

customer_rfm["Segment_Detailed"] = [assign_detailed_segment(r, fm)
                                    for r, fm in zip(customer_rfm["R_Score"], customer_rfm["FM_Score"])]
customer_rfm["Segment"] = customer_rfm["Segment_Detailed"].map(SEGMENT_MAP)

present = [s for s in SEGMENT_ORDER if s in set(customer_rfm["Segment"])]
detailed_present = [s for s in DETAILED_ORDER if s in set(customer_rfm["Segment_Detailed"])]


def summarize_segments(group_col, order):
    out = (
        customer_rfm.groupby(group_col)
        .agg(
            Customers=("CustomerID", "count"),
            Avg_Recency=("Recency", "mean"),
            Avg_Frequency=("Frequency", "mean"),
            Avg_Monetary=("Monetary", "mean"),
            Median_Monetary=("Monetary", "median"),
            Total_Revenue=("Monetary", "sum"),
            Total_Invoices=("Frequency", "sum"),
        )
        .loc[order]
        .reset_index()
    )
    out["Customers_%"] = out["Customers"] / out["Customers"].sum() * 100
    out["Revenue_%"] = out["Total_Revenue"] / out["Total_Revenue"].sum() * 100
    out["Avg_Order_Value"] = out["Total_Revenue"] / out["Total_Invoices"]
    return out

# Main segments (4) and sub-segments (11)
segment_summary = summarize_segments("Segment", present)
detailed_summary = summarize_segments("Segment_Detailed", detailed_present)
detailed_summary.insert(0, "Segment", detailed_summary["Segment_Detailed"].map(SEGMENT_MAP))

print("=== 4 main segments ===")
display(segment_summary.round(1))
print("=== 11 sub-segments (grouped under their main segment) ===")
display(detailed_summary.round(1))

=== 4 main segments ===


,Segment,Customers,Avg_Recency,Avg_Frequency,Avg_Monetary,Median_Monetary,Total_Revenue,Total_Invoices,Customers_%,Revenue_%,Avg_Order_Value
0,Champions,655,5.9,12.0,6900.3,2567.1,4519673.8,7848,15.1,51.6,575.9
1,Loyal,1454,27.2,4.4,1855.2,990.6,2697394.3,6412,33.5,30.8,420.7
2,At Risk,1273,115.7,2.4,1024.0,609.8,1303595.2,3106,29.4,14.9,419.7
3,Lost,953,221.4,1.1,257.9,225.6,245793.9,1039,22.0,2.8,236.6


=== 11 sub-segments (grouped under their main segment) ===


,Segment,Segment_Detailed,Customers,Avg_Recency,Avg_Frequency,Avg_Monetary,Median_Monetary,Total_Revenue,Total_Invoices,Customers_%,Revenue_%,Avg_Order_Value
0,Champions,Champions,655,5.9,12.0,6900.3,2567.1,4519673.8,7848,15.1,51.6,575.9
1,Loyal,Loyal Customers,890,33.7,6.1,2748.0,1704.4,2445720.4,5423,20.5,27.9,451.0
2,Loyal,Potential Loyalists,491,16.8,1.9,489.1,408.8,240156.9,916,11.3,2.7,262.2
3,Loyal,New Customers,21,6.5,1.0,180.2,194.4,3784.6,21,0.5,0.0,180.2
4,Loyal,Promising,52,22.9,1.0,148.7,150.2,7732.5,52,1.2,0.1,148.7
5,At Risk,Need Attention,198,51.7,2.0,692.4,607.2,137088.3,403,4.6,1.6,340.2
6,At Risk,About to Sleep,294,52.6,1.1,278.8,239.7,81979.2,328,6.8,0.9,249.9
7,At Risk,At Risk,691,160.2,2.5,1119.9,716.4,773844.1,1760,15.9,8.8,439.7
8,At Risk,Can't Lose Them,90,122.2,6.8,3452.0,2521.2,310683.7,615,2.1,3.5,505.2
9,Lost,Hibernating,774,206.2,1.1,284.5,271.5,220219.4,860,17.9,2.5,256.1


### 8.8 Visualizing the Segments

The charts below answer different questions:

| # | Chart | Question it answers |
|---|---|---|
| 1 | RFM grid | Where do customers sit on the R × FM grid, and which main segment does each cell belong to? |
| 2 | Customers vs revenue | What share of the customers and of the revenue does each segment hold? |
| 3 | Treemap | How big is each segment, and what is inside it? (click a block to zoom in) |
| 4 | Scatter (2D / 3D) | What does the customer cloud look like in R, F and M space? |
| 5 | Profile heatmap & box plots | What is the typical R / F / M score and raw value of each segment? |
| 6 | Lorenz curve | How concentrated is the revenue among customers? |

**Why filters?** A couple of things dominate these charts: the **Champions** segment holds most of the revenue, and a handful of **huge customers** stretch every axis. That makes small segments almost invisible. Every interactive chart therefore has **dropdown menus** above the plot:

- **Segment filter** — show all segments, one segment only, or hide the Champions so the rest can be compared at their own scale.
- **Axis scale** — switch between log and linear, or zoom to the 99th percentile to cut off the extreme outliers.
- **View / value** — switch between the 4 main segments and the 11 sub-segments, or between customer counts and revenue.

Tip: you can also click an entry in the legend to hide or show a segment, and double-click it to isolate it.

**Helper functions for the dropdown filters.** `add_dropdown` attaches a dropdown menu to a figure, and `segment_filter_buttons` builds the "All / Only X / Hide Champions" buttons by switching the `visible` flag of each segment's trace. The charts below simply reuse them.

In [94]:
def add_dropdown(fig, buttons, x=0.0, y=1.0):
    """Attach one more dropdown menu above the plot (keeps any menus already on the figure)."""
    menu = dict(type="dropdown", buttons=buttons, x=x, xanchor="left", y=y, yanchor="bottom",
                showactive=True, bgcolor="white", bordercolor="#BDC3C7")
    fig.update_layout(updatemenus=list(fig.layout.updatemenus) + [menu], margin=dict(t=130))


def segment_filter_buttons(fig, segments):
    """Buttons: All segments / Only <segment> / Hide Champions (toggles trace visibility by trace name)."""
    names = [t.name for t in fig.data]
    def visible(keep):
        return [n in keep for n in names]
    buttons = [dict(label="All segments", method="update", args=[{"visible": visible(set(segments))}])]
    buttons += [dict(label=f"Only {s}", method="update", args=[{"visible": visible({s})}]) for s in segments]
    buttons.append(dict(label="Hide Champions", method="update",
                        args=[{"visible": visible(set(segments) - {"Champions"})}]))
    return buttons

#### 8.8.1 RFM Grid

Each cell of the grid is one (R, FM) combination. The **text** shows the sub-segment name and how many customers fall in it; the **colour** shows the **main segment** it rolls up into (see the legend). This is the picture that connects the 11 sub-segments to the 4 main segments.

In [95]:
# RFM grid: customers per (R, FM) cell, coloured by main segment, labelled with the sub-segment
grid_counts = (pd.crosstab(customer_rfm["R_Score"], customer_rfm["FM_Score"])
               .reindex(index=range(1, 6), columns=range(1, 6), fill_value=0))

group_index = {seg: i for i, seg in enumerate(SEGMENT_ORDER)}
z = [[group_index[SEGMENT_MAP[segment_grid[r][fm - 1]]] for fm in range(1, 6)] for r in range(1, 6)]
text = [[f"<b>{segment_grid[r][fm - 1]}</b><br>{grid_counts.loc[r, fm]:,} customers" for fm in range(1, 6)]
        for r in range(1, 6)]
main_seg = [[SEGMENT_MAP[segment_grid[r][fm - 1]] for fm in range(1, 6)] for r in range(1, 6)]

# discrete colour scale: one flat colour band per main segment
n = len(SEGMENT_ORDER)
colorscale = []
for i, seg in enumerate(SEGMENT_ORDER):
    colorscale += [[i / n, SEGMENT_COLORS[seg]], [(i + 1) / n, SEGMENT_COLORS[seg]]]

fig = go.Figure(go.Heatmap(
    z=z, x=[1, 2, 3, 4, 5], y=[1, 2, 3, 4, 5], text=text, texttemplate="%{text}",
    textfont=dict(color="white", size=12), customdata=main_seg,
    colorscale=colorscale, zmin=-0.5, zmax=n - 0.5, showscale=False, xgap=3, ygap=3,
    hovertemplate="R=%{y}, FM=%{x}<br>Main segment: %{customdata}<extra></extra>",
))
# dummy traces, only to draw a legend for the main segments
for seg in SEGMENT_ORDER:
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=seg,
                             marker=dict(size=12, symbol="square", color=SEGMENT_COLORS[seg])))
fig.update_layout(title="RFM Grid: where customers sit, and which main segment each cell belongs to", title_x=0.5,
                  xaxis=dict(title="FM_Score (Frequency + Monetary)", dtick=1),
                  yaxis=dict(title="R_Score (Recency)", dtick=1), height=560, legend_title="Main segment")
fig.show()

#### 8.8.2 Customers vs Revenue, and the Treemap

**Share of customers vs share of revenue.** For each segment, the grey bar is the percentage of all customers and the green bar the percentage of all revenue. A segment whose green bar is much taller than its grey bar is carrying more than its weight (typically the Champions); the opposite means many customers but little money (typically Lost).

Use the dropdown to **switch between the 4 main segments and the 11 sub-segments**, and to **hide the Champions** — their bar is so tall that it flattens everything else.

**Treemap.** The area of each block is the number of customers (or the revenue, using the dropdown). The first level is the main segment and the second level is the sub-segment. **Click a main segment to zoom into it**; click the top bar to go back.

In [96]:
# ---- Share of customers vs share of revenue (with view filter) ----
def share_view(df, name_col, hide=()):
    d = df[~df[name_col].isin(hide)]
    return d[name_col].tolist(), d["Customers_%"].round(1).tolist(), d["Revenue_%"].round(1).tolist()

views = {
    "4 main segments":                      share_view(segment_summary, "Segment"),
    "4 main segments - hide Champions":     share_view(segment_summary, "Segment", ("Champions",)),
    "11 sub-segments":                      share_view(detailed_summary, "Segment_Detailed"),
    "11 sub-segments - hide Champions":     share_view(detailed_summary, "Segment_Detailed", ("Champions",)),
}
xs, cust_pct, rev_pct = views["4 main segments"]

fig = go.Figure([
    go.Bar(name="% of customers", x=xs, y=cust_pct, marker_color="#95A5A6", texttemplate="%{y:.1f}"),
    go.Bar(name="% of revenue",   x=xs, y=rev_pct,  marker_color="#1B7F4C", texttemplate="%{y:.1f}"),
])
buttons = [dict(label=name, method="restyle", args=[{"x": [v[0], v[0]], "y": [v[1], v[2]]}])
           for name, v in views.items()]
fig.update_layout(barmode="group", title="Share of customers vs share of revenue, by segment", title_x=0.5,
                  height=500, yaxis_title="Share (%)", legend_title="")
add_dropdown(fig, buttons, x=0.0)
fig.show()


# ---- Treemap: size of each segment (switch between customers and revenue) ----
tree_df = detailed_summary.rename(columns={"Segment": "Main segment", "Segment_Detailed": "Sub-segment"})

def make_treemap(value_col):
    f = px.treemap(tree_df, path=[px.Constant("All customers"), "Main segment", "Sub-segment"],
                   values=value_col, color="Main segment",
                   color_discrete_map={**SEGMENT_COLORS, "(?)": "#FFFFFF"})
    f.update_traces(textinfo="label+value+percent root", root_color="white")
    return f

fig = make_treemap("Customers")
rev_trace = make_treemap("Total_Revenue").data[0]
rev_trace.visible = False
fig.add_trace(rev_trace)

fig.update_layout(title="Customer segments (area = number of customers; click a block to zoom)", title_x=0.5,
                  height=520, margin=dict(t=130))
add_dropdown(fig, [
    dict(label="Area = number of customers", method="update",
         args=[{"visible": [True, False]}, {"title.text": "Customer segments (area = number of customers; click a block to zoom)"}]),
    dict(label="Area = revenue", method="update",
         args=[{"visible": [False, True]}, {"title.text": "Customer segments (area = revenue; click a block to zoom)"}]),
], x=0.0)
fig.show()

#### 8.8.3 Scatter Plots — 2D and 3D

**2D (Frequency vs Monetary).** One dot per customer, coloured by main segment. Because the dots overlap heavily, use the filters:

- **Segment filter** — isolate one segment, or hide the Champions to see the others clearly.
- **Axis scale** — *Log-log* is the default and spreads the dots out. *Linear* shows the real proportions (and how extreme the biggest customers are). *Linear, zoom to 99th percentile* cuts off the top 1% of customers on each axis so the bulk of the customers becomes readable.

**3D (Recency, log10 Frequency, log10 Monetary).** The same cloud in all three RFM dimensions at once. Drag to rotate; use the segment filter to see where one segment sits relative to the others.

In [97]:
# ---- 2D view: Frequency vs Monetary ----
fig = px.scatter(
    customer_rfm, x="Frequency", y="Monetary", color="Segment",
    category_orders={"Segment": present}, color_discrete_map=SEGMENT_COLORS,
    hover_data=["CustomerID", "Segment_Detailed", "Recency", "RFM_Score"],
    log_x=True, log_y=True, opacity=0.7,
    title="Customer segments: Frequency vs Monetary",
)
fig.update_layout(title_x=0.5, height=580)

f99 = customer_rfm["Frequency"].quantile(0.99)
m99 = customer_rfm["Monetary"].quantile(0.99)
scale_buttons = [
    dict(label="Log-log (default)", method="relayout",
         args=[{"xaxis.type": "log", "yaxis.type": "log", "xaxis.autorange": True, "yaxis.autorange": True}]),
    dict(label="Linear", method="relayout",
         args=[{"xaxis.type": "linear", "yaxis.type": "linear", "xaxis.autorange": True, "yaxis.autorange": True}]),
    dict(label="Linear, zoom to 99th percentile", method="relayout",
         args=[{"xaxis.type": "linear", "yaxis.type": "linear",
                "xaxis.range": [0, float(f99)], "yaxis.range": [0, float(m99)]}]),
]
add_dropdown(fig, segment_filter_buttons(fig, present), x=0.0)
add_dropdown(fig, scale_buttons, x=0.27)
fig.show()


# ---- 3D view: Recency, Frequency, Monetary together (F and M on log10 scale) ----
rfm_3d = customer_rfm.assign(log10_Frequency=np.log10(customer_rfm["Frequency"]),
                             log10_Monetary=np.log10(customer_rfm["Monetary"]))
fig = px.scatter_3d(
    rfm_3d, x="Recency", y="log10_Frequency", z="log10_Monetary", color="Segment",
    category_orders={"Segment": present}, color_discrete_map=SEGMENT_COLORS,
    hover_data=["CustomerID", "Segment_Detailed"], opacity=0.75,
    title="Customers in RFM space (3D)",
)
fig.update_traces(marker_size=3)
fig.update_layout(title_x=0.5, height=640)
add_dropdown(fig, segment_filter_buttons(fig, present), x=0.0)
fig.show()

#### 8.8.4 Segment Profiles — Average Scores and Raw Distributions

**Profile heatmap.** The average R, F and M **score** (1–5) of each segment. Green = strong, red = weak. It is a fingerprint of the segment: Champions should be green everywhere, Lost should be red everywhere, and At Risk should show a good F/M but a poor R. Use the dropdown to switch between main segments and sub-segments.

**Box plots.** The same comparison using the **raw values**, one chart each for Recency, Frequency and Monetary. A box covers the middle 50% of customers, the line inside is the median, and the dots are outliers. Frequency and Monetary start on a log scale. Each chart has three filters:

- **Segment filter** — isolate a segment or hide the Champions.
- **Outliers** — hide the outlier dots to zoom in on the boxes themselves.
- **Scale** (Frequency and Monetary only) — switch between log and linear.

In [98]:
# ---- Profile heatmap: average R / F / M score per segment (main <-> sub-segments) ----
score_cols = ["R_Score", "F_Score", "M_Score"]
prof_main = customer_rfm.groupby("Segment")[score_cols].mean().loc[present].round(2)
prof_detail = customer_rfm.groupby("Segment_Detailed")[score_cols].mean().loc[detailed_present].round(2)

fig = go.Figure(go.Heatmap(
    z=prof_main.values.tolist(), x=score_cols, y=prof_main.index.tolist(),
    text=prof_main.values.tolist(), texttemplate="%{text:.2f}",
    zmin=1, zmax=5, colorscale="RdYlGn", colorbar=dict(title="Avg score"),
))
fig.update_yaxes(autorange="reversed")
fig.update_layout(title="Average R / F / M score per segment", title_x=0.5, height=520)
add_dropdown(fig, [
    dict(label="4 main segments", method="restyle",
         args=[{"z": [prof_main.values.tolist()], "y": [prof_main.index.tolist()], "text": [prof_main.values.tolist()]}]),
    dict(label="11 sub-segments", method="restyle",
         args=[{"z": [prof_detail.values.tolist()], "y": [prof_detail.index.tolist()], "text": [prof_detail.values.tolist()]}]),
], x=0.0)
fig.show()


# ---- Box plots: raw values per segment, with filters ----
for col, use_log in [("Recency", False), ("Frequency", True), ("Monetary", True)]:
    fig = px.box(customer_rfm, x="Segment", y=col, color="Segment", points="outliers",
                 category_orders={"Segment": present}, color_discrete_map=SEGMENT_COLORS,
                 log_y=use_log, title=f"{col} by segment" + (" (log scale)" if use_log else ""))
    fig.update_layout(title_x=0.5, showlegend=False, height=500, xaxis_title="")

    add_dropdown(fig, segment_filter_buttons(fig, present), x=0.0)
    add_dropdown(fig, [
        dict(label="Show outliers", method="restyle", args=[{"boxpoints": "outliers"}]),
        dict(label="Hide outliers (zoom in)", method="restyle", args=[{"boxpoints": False}]),
    ], x=0.27)
    if use_log:   # log / linear switch only makes sense for the skewed metrics
        add_dropdown(fig, [
            dict(label="Log scale", method="relayout", args=[{"yaxis.type": "log", "yaxis.autorange": True}]),
            dict(label="Linear scale", method="relayout", args=[{"yaxis.type": "linear", "yaxis.autorange": True}]),
        ], x=0.58)
    fig.show()

#### 8.8.5 Revenue Concentration (Lorenz-style curve)

Customers are sorted from the biggest spender to the smallest. The green curve shows how much of the total revenue the best X% of customers generate; the dotted diagonal is what a perfectly equal business would look like. The further the curve bulges above the diagonal, the more the business depends on a few customers. The printed line gives the share of revenue produced by the top 20% of customers.

In [99]:
# How concentrated is revenue among customers?
rfm_sorted = customer_rfm.sort_values("Monetary", ascending=False).reset_index(drop=True)
rfm_sorted["Cum_Customer_Share"] = np.arange(1, len(rfm_sorted) + 1) / len(rfm_sorted)
rfm_sorted["Cum_Revenue_Share"] = rfm_sorted["Monetary"].cumsum() / rfm_sorted["Monetary"].sum()

top20_share = rfm_sorted.loc[rfm_sorted["Cum_Customer_Share"] <= 0.20, "Cum_Revenue_Share"].iloc[-1]
print(f"Top 20% of customers generate {top20_share:.1%} of revenue")

fig = go.Figure()
fig.add_trace(go.Scatter(x=rfm_sorted["Cum_Customer_Share"], y=rfm_sorted["Cum_Revenue_Share"],
                         mode="lines", fill="tozeroy", name="Cumulative revenue share",
                         line=dict(color="#1B7F4C", width=3)))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Perfect equality",
                         line=dict(color="#95A5A6", dash="dot")))
fig.update_layout(title="Customer revenue concentration (Lorenz-style curve)", title_x=0.5,
                  xaxis=dict(title="Cumulative share of customers (best first)", tickformat=".0%"),
                  yaxis=dict(title="Cumulative share of revenue", tickformat=".0%"),
                  hovermode="x unified", height=450)
fig.show()

Top 20% of customers generate 74.5% of revenue


### 8.9 Sensitivity Check — Does Keeping the Duplicates Change the Result?

In section 6.2 I flagged duplicated transaction lines but did **not** delete them. This check is the promised proof that this decision is harmless. The RFM is computed twice:

- **A:** as is — the duplicates are kept (this is what the whole section uses).
- **B:** with the flagged duplicates removed — used **only for comparison**, the original data is untouched.

Recency and Frequency must be **identical** in both versions (a duplicated line does not change the last purchase date or the number of invoices), so the effect can only show up in **Monetary**, and from there in the scores and segments. If only a tiny share of customers change segment, then keeping the duplicates did not affect the conclusions. The segment comparison below is made on the **4 main segments**.

In [100]:
# Sensitivity check: RFM with vs without the flagged duplicate lines
def quintile_score(values, reverse=False):
    labels = [5, 4, 3, 2, 1] if reverse else [1, 2, 3, 4, 5]
    return pd.qcut(values.rank(method="first"), q=5, labels=labels).astype(int)

tx_no_dups = customer_transactions[~customer_transactions["Is_Duplicate_Line"]]

rfm_b = (
    tx_no_dups.groupby("CustomerID")
    .agg(Recency=("InvoiceDate", "max"),
         Frequency=("InvoiceNo", "nunique"),
         Monetary=("TotalLineRevenue", "sum"))
    .reset_index()
)
rfm_b["Recency"] = (snapshot_date - rfm_b["Recency"]).dt.days
rfm_b["R_Score"] = quintile_score(rfm_b["Recency"], reverse=True)
rfm_b["F_Score"] = quintile_score(rfm_b["Frequency"])
rfm_b["M_Score"] = quintile_score(rfm_b["Monetary"])
rfm_b["FM_Score"] = np.ceil((rfm_b["F_Score"] + rfm_b["M_Score"]) / 2).astype(int)
rfm_b["Segment"] = [assign_segment(r, fm) for r, fm in zip(rfm_b["R_Score"], rfm_b["FM_Score"])]

cmp = customer_rfm[["CustomerID", "Recency", "Frequency", "Monetary", "R_Score", "F_Score", "M_Score", "Segment"]].merge(
    rfm_b[["CustomerID", "Recency", "Frequency", "Monetary", "R_Score", "F_Score", "M_Score", "Segment"]],
    on="CustomerID", suffixes=("_A", "_B"))

sens = pd.DataFrame({
    "Metric": [
        "Customers with different Recency",
        "Customers with different Frequency",
        "Customers with different Monetary",
        "Customers whose M_Score changed",
        "Customers whose Segment changed",
        "Total revenue difference (A vs B)",
    ],
    "Value": [
        f"{(cmp['Recency_A'] != cmp['Recency_B']).sum():,}",
        f"{(cmp['Frequency_A'] != cmp['Frequency_B']).sum():,}",
        f"{(~np.isclose(cmp['Monetary_A'], cmp['Monetary_B'])).sum():,} ({(~np.isclose(cmp['Monetary_A'], cmp['Monetary_B'])).mean():.1%})",
        f"{(cmp['M_Score_A'] != cmp['M_Score_B']).sum():,} ({(cmp['M_Score_A'] != cmp['M_Score_B']).mean():.1%})",
        f"{(cmp['Segment_A'] != cmp['Segment_B']).sum():,} ({(cmp['Segment_A'] != cmp['Segment_B']).mean():.1%})",
        f"{(cmp['Monetary_A'].sum() - cmp['Monetary_B'].sum()):,.0f} ({(cmp['Monetary_A'].sum() / cmp['Monetary_B'].sum() - 1):.2%})",
    ],
})
display(sens)

# Segment sizes side by side
seg_cmp = pd.DataFrame({
    "A: keep duplicates": cmp["Segment_A"].value_counts(),
    "B: remove duplicates": cmp["Segment_B"].value_counts(),
}).reindex(present).fillna(0).astype(int).reset_index().rename(columns={"index": "Segment"})
seg_cmp = seg_cmp.rename(columns={seg_cmp.columns[0]: "Segment"})

fig = px.bar(seg_cmp.melt(id_vars="Segment", var_name="Version", value_name="Customers"),
             x="Segment", y="Customers", color="Version", barmode="group",
             category_orders={"Segment": present},
             color_discrete_map={"A: keep duplicates": "#2E86DE", "B: remove duplicates": "#E67E22"},
             title="Segment sizes: keeping vs removing the duplicate lines")
fig.update_layout(title_x=0.5, height=430, legend_title="")
fig.show()

,Metric,Value
0,Customers with different Recency,0
1,Customers with different Frequency,0
2,Customers with different Monetary,952 (22.0%)
3,Customers whose M_Score changed,38 (0.9%)
4,Customers whose Segment changed,2 (0.0%)
5,Total revenue difference (A vs B),"24,391 (0.28%)"


**Key findings.** The cell below prints, for each main segment, its number of customers and its share of revenue, and then the numbers behind the two groups the business should worry about most: the high-value customers who are drifting away (*Can't Lose Them*) and the customers that are already gone.

In [101]:
seg = segment_summary.set_index("Segment")
det = detailed_summary.set_index("Segment_Detailed")

print("Main segments: customers -> share of revenue")
for s in present:
    print(f"  {s:<10}: {seg.loc[s, 'Customers']:>5,.0f} customers ({seg.loc[s, 'Customers_%']:4.1f}%) "
          f"-> {seg.loc[s, 'Revenue_%']:4.1f}% of revenue")

active = seg.loc[["Champions", "Loyal"]]
print(f"\nChampions + Loyal      : {active['Customers_%'].sum():.1f}% of customers -> {active['Revenue_%'].sum():.1f}% of revenue (the core)")

vip_name = "Can't Lose Them"
vip = det.loc[vip_name]
print(f"{vip_name:<23}: {vip['Customers']:.0f} customers ({vip['Customers_%']:.1f}%) -> {vip['Revenue_%']:.1f}% of revenue "
      f"(inside At Risk; highest win-back priority)")

gone = seg.loc["Lost"]
print(f"Lost                   : {gone['Customers_%']:.1f}% of customers -> {gone['Revenue_%']:.1f}% of revenue (lowest priority)")

Main segments: customers -> share of revenue
  Champions :   655 customers (15.1%) -> 51.6% of revenue
  Loyal     : 1,454 customers (33.5%) -> 30.8% of revenue
  At Risk   : 1,273 customers (29.4%) -> 14.9% of revenue
  Lost      :   953 customers (22.0%) ->  2.8% of revenue

Champions + Loyal      : 48.7% of customers -> 82.3% of revenue (the core)
Can't Lose Them        : 90 customers (2.1%) -> 3.5% of revenue (inside At Risk; highest win-back priority)
Lost                   : 22.0% of customers -> 2.8% of revenue (lowest priority)


### 8.10 Business Interpretation

The numbers for each segment (size and share of revenue) are in the tables and charts above. The table below connects every segment to a suggested action — first the 4 main segments, then the sub-segments inside each.

| Main segment | Sub-segment | What it means | Suggested action |
|---|---|---|---|
| **Champions** | Champions | Bought recently, buy often and spend a lot | Reward them, offer early access to new products, ask for reviews and referrals |
| **Loyal** | Loyal Customers | Buy regularly | Loyalty programme, upsell / cross-sell offers |
| | Potential Loyalists | Recent, medium value | Incentives that encourage repeat purchases |
| | New Customers | First purchase was recent | Onboarding experience, nudge them to a quick second purchase |
| | Promising | Recent but low spend | Small offers to raise basket size |
| **At Risk** | Need Attention | Mid-range, due for a reminder | Limited-time offer |
| | About to Sleep | Starting to drift away | Remind them of products close to their last purchase |
| | At Risk | Were good customers, now fading | Personalised win-back campaign |
| | Can't Lose Them | Very high value but gone quiet | Direct personal contact (account manager) |
| **Lost** | Hibernating | Away for a while, low value | Low-cost discount campaign |
| | Lost | Away for very long, low value | Lowest priority; one last email attempt |

> These actions are business suggestions, not results that come from the data itself.

## 9. Product-Level RFM Analysis

The same idea, but at the **product** level (`StockCode`):
- **Recency:** how many days since the product was last sold?
- **Frequency:** in how many unique invoices did it appear?
- **Monetary:** the total revenue it brought in.

I also added columns that help with interpretation: `Units_Sold`, `Unique_Customers`, `First_Sale`, `Active_Days`, `Avg_Price`.

### 9.1 Which Transactions?

For products I used all valid sales (`Quantity > 0`, `UnitPrice > 0` and not a cancellation), **including Guests**, because a product was sold whether or not the buyer is registered. Excluding the Guests would be unfair to products that are mostly bought by unregistered customers (about a quarter of the rows). This is a difference from the older version, which computed on `customer_transactions` only.

In [102]:
# All valid sales lines, Guests included (a product is sold whether or not the buyer is registered)
product_transactions = df_products_clean[
    (df_products_clean["Quantity"] > 0) &
    (df_products_clean["UnitPrice"] > 0) &
    (df_products_clean["TotalLineRevenue"] > 0) &
    (~df_products_clean["InvoiceNo"].astype(str).str.startswith("C")) &
    (df_products_clean["InvoiceDate"].notna())
].copy()

print("Product transactions:", f"{len(product_transactions):,}")
print("Unique products     :", f"{product_transactions['StockCode'].nunique():,}")
print("Rows from Guests    :", f"{product_transactions['CustomerID'].isna().sum():,}",
      f"({product_transactions['CustomerID'].isna().mean():.1%})")

Product transactions: 528,913
Unique products     : 3,912
Rows from Guests    : 132,539 (25.1%)


### 9.2 Building R, F, M per Product

I use the same `snapshot_date` as for the customers, so that product Recency and customer Recency are measured from the same point in time.

In [103]:
product_rfm = (
    product_transactions
    .groupby("StockCode")
    .agg(
        Last_Sale=("InvoiceDate", "max"),
        First_Sale=("InvoiceDate", "min"),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("TotalLineRevenue", "sum"),
        Units_Sold=("Quantity", "sum"),
        Unique_Customers=("CustomerID", "nunique"),
        Description=("Description", "first"),
    )
    .reset_index()
)

product_rfm["Recency"] = (snapshot_date - product_rfm["Last_Sale"]).dt.days
product_rfm["Active_Days"] = (product_rfm["Last_Sale"] - product_rfm["First_Sale"]).dt.days
product_rfm["Avg_Price"] = product_rfm["Monetary"] / product_rfm["Units_Sold"]
product_rfm["Product_Label"] = (
    product_rfm["StockCode"].astype(str) + " - " +
    product_rfm["Description"].fillna("No description").astype(str).str.slice(0, 35)
)

display(product_rfm[["StockCode", "Description", "Recency", "Frequency", "Monetary",
                     "Units_Sold", "Unique_Customers", "Active_Days", "Avg_Price"]].head(10))
display(product_rfm[["Recency", "Frequency", "Monetary", "Units_Sold"]].describe().T)

,StockCode,Description,Recency,Frequency,Monetary,Units_Sold,Unique_Customers,Active_Days,Avg_Price
0,10002,INFLATABLE POLITICAL GLOBE,235,72,912.89,1040,40,138,0.877779
1,10080,GROOVY CACTUS INFLATABLE,18,24,193.97,495,19,267,0.391859
2,10120,DOGGY RUBBER,5,29,40.53,193,25,366,0.210000
3,10123C,HEARTS WRAPPING TAPE,253,3,3.25,5,3,118,0.650000
4,10124A,SPOTS ON RED BOOKCOVER TAPE,33,5,6.72,16,5,334,0.420000
5,10124G,ARMY CAMO BOOKCOVER TAPE,33,4,7.14,17,4,338,0.420000
6,10125,MINI FUNKY DESIGN TAPES,1,91,994.84,1296,49,372,0.767623
7,10133,COLOURING PENCILS BROWN TUBE,94,196,1544.22,2867,101,279,0.538619
8,10135,COLOURING PENCILS BROWN TUBE,2,175,2207.39,2231,92,371,0.989417
9,11001,ASSTD DESIGN RACING CAR PEN,3,113,2389.44,1615,44,370,1.479529


,count,mean,std,min,25%,50%,75%,max
Recency,3912.0,46.492587,90.271003,1.000,1.0000,3.000,30.00,374.00
Frequency,3912.0,132.518405,193.320070,1.000,16.0000,65.000,165.00,2203.00
Monetary,3912.0,2664.848751,7225.184509,0.004,135.0925,698.665,2262.74,174765.13
Units_Sold,3912.0,1442.623466,3614.123911,1.000,59.0000,384.500,1409.50,80995.00


### 9.3 Distributions

As with customers, Frequency and Monetary are shown on a log10 scale because revenue is concentrated in a small share of the products (as the Pareto analysis in 7.8 showed).

In [104]:
product_dist_specs = [
    ("Recency",   "Product Recency: days since last sale",         False),
    ("Frequency", "Product Frequency: unique invoices (log10)",    True),
    ("Monetary",  "Product Monetary: total revenue (log10)",       True),
]

for col, title, use_log in product_dist_specs:
    plot_df = product_rfm[["StockCode", col]].copy()
    x_col = col
    if use_log:
        x_col = f"log10_{col}"
        plot_df[x_col] = np.log10(plot_df[col])

    fig = px.histogram(plot_df, x=x_col, nbins=50, marginal="box",
                       title=title, color_discrete_sequence=["#8E44AD"])
    fig.update_layout(title_x=0.5, height=420, bargap=0.05)
    fig.show()

### 9.4 Product Scoring

The same idea: quintiles from 1 to 5 using `rank(method="first")`, with R reversed (the most recently sold product = 5). Then a heatmap shows the **median revenue** of every R × F combination, to make sure the scores really separate products by value.

In [105]:
product_rfm["R_Score"] = quintile_score(product_rfm["Recency"], reverse=True)
product_rfm["F_Score"] = quintile_score(product_rfm["Frequency"])
product_rfm["M_Score"] = quintile_score(product_rfm["Monetary"])
product_rfm["RFM_Score"] = (product_rfm["R_Score"].astype(str) +
                            product_rfm["F_Score"].astype(str) +
                            product_rfm["M_Score"].astype(str))

# Median revenue for each R x F combination
rf_value = product_rfm.pivot_table(index="R_Score", columns="F_Score", values="Monetary",
                                   aggfunc="median").reindex(index=range(1, 6), columns=range(1, 6))
fig = px.imshow(rf_value, text_auto=",.0f", aspect="auto", origin="lower",
                color_continuous_scale="Purples",
                labels=dict(x="F_Score (Frequency)", y="R_Score (Recency)", color="Median revenue"),
                title="Median product revenue by Recency score and Frequency score")
fig.update_layout(title_x=0.5, height=450)
fig.show()

### 9.5 Product Segments

The rules are listed in priority order (the first condition that is met decides the segment):

| Segment | Rule | Meaning |
|---|---|---|
| Star Products | R ≥ 4 and F ≥ 4 and M ≥ 4 | Selling now, often, and with high revenue |
| New Arrivals | First sale within the last 60 days | A new product, too early to judge |
| Core Sellers | R ≥ 3 and F ≥ 3 and M ≥ 3 | The backbone of the store |
| Fading Products | R ≤ 2 and F ≥ 3 and M ≥ 3 | Was strong and then stopped or slowed down (may be seasonal) |
| Niche High-Value | M ≥ 4 and F ≤ 3 | Sells rarely but with high revenue (usually large quantities) |
| Volume Drivers | F ≥ 4 and M ≤ 3 | Appears in many invoices but brings little revenue (cheap price) |
| Dead Stock | R ≤ 2 and F ≤ 2 and M ≤ 2 | Rare and old, a decision is needed (clearance / discontinue) |
| Occasional Sellers | Anything else | Intermittent sales |

> **Warning:** the data covers one year only, so a seasonal product (such as Christmas decorations) can come out as Fading or Dead Stock when it is merely out of season. The result therefore needs review before any decision to remove a product from the catalogue.

In [106]:
r, f, m = product_rfm["R_Score"], product_rfm["F_Score"], product_rfm["M_Score"]
new_cutoff = snapshot_date - pd.Timedelta(days=60)

conditions = [
    (r >= 4) & (f >= 4) & (m >= 4),               # Star Products
    product_rfm["First_Sale"] >= new_cutoff,      # New Arrivals
    (r >= 3) & (f >= 3) & (m >= 3),               # Core Sellers
    (r <= 2) & (f >= 3) & (m >= 3),               # Fading Products
    (m >= 4) & (f <= 3),                          # Niche High-Value
    (f >= 4) & (m <= 3),                          # Volume Drivers
    (r <= 2) & (f <= 2) & (m <= 2),               # Dead Stock
]
choices = ["Star Products", "New Arrivals", "Core Sellers", "Fading Products",
           "Niche High-Value", "Volume Drivers", "Dead Stock"]
product_rfm["Segment"] = np.select(conditions, choices, default="Occasional Sellers")

PRODUCT_SEGMENT_ORDER = ["Star Products", "Core Sellers", "New Arrivals", "Niche High-Value",
                         "Volume Drivers", "Occasional Sellers", "Fading Products", "Dead Stock"]
PRODUCT_SEGMENT_COLORS = {
    "Star Products": "#1B7F4C", "Core Sellers": "#4CAF7A", "New Arrivals": "#2E86DE",
    "Niche High-Value": "#8E44AD", "Volume Drivers": "#F5B041", "Occasional Sellers": "#95A5A6",
    "Fading Products": "#E67E22", "Dead Stock": "#566573",
}
p_present = [s for s in PRODUCT_SEGMENT_ORDER if s in set(product_rfm["Segment"])]

product_segment_summary = (
    product_rfm.groupby("Segment")
    .agg(
        Products=("StockCode", "count"),
        Total_Revenue=("Monetary", "sum"),
        Avg_Recency=("Recency", "mean"),
        Avg_Frequency=("Frequency", "mean"),
        Median_Price=("Avg_Price", "median"),
        Units_Sold=("Units_Sold", "sum"),
    )
    .loc[p_present]
    .reset_index()
)
product_segment_summary["Products_%"] = product_segment_summary["Products"] / product_segment_summary["Products"].sum() * 100
product_segment_summary["Revenue_%"] = product_segment_summary["Total_Revenue"] / product_segment_summary["Total_Revenue"].sum() * 100

display(product_segment_summary.round(1))

share_long = product_segment_summary.melt(id_vars="Segment", value_vars=["Products_%", "Revenue_%"],
                                          var_name="Metric", value_name="Share_%")
fig = px.bar(share_long, x="Segment", y="Share_%", color="Metric", barmode="group",
             category_orders={"Segment": p_present},
             color_discrete_map={"Products_%": "#95A5A6", "Revenue_%": "#8E44AD"},
             text_auto=".1f", title="Share of products vs share of revenue, by product segment")
fig.update_layout(title_x=0.5, height=470, yaxis_title="Share (%)", legend_title="")
fig.show()

,Segment,Products,Total_Revenue,Avg_Recency,Avg_Frequency,Median_Price,Units_Sold,Products_%,Revenue_%
0,Star Products,988,7609557.8,1.0,347.8,1.9,3946925,25.3,73.0
1,Core Sellers,769,1437083.6,2.1,126.1,1.7,925040,19.7,13.8
2,New Arrivals,118,239019.6,10.1,22.6,2.5,99957,3.0,2.3
3,Niche High-Value,44,122936.5,83.7,25.7,5.0,39447,1.1,1.2
4,Volume Drivers,18,6250.9,2.1,114.5,0.4,18651,0.5,0.1
5,Occasional Sellers,630,183115.8,24.2,32.0,1.9,161580,16.1,1.8
6,Fading Products,365,738710.1,59.0,114.4,1.9,379638,9.3,7.1
7,Dead Stock,980,88213.9,140.4,10.2,2.5,72305,25.1,0.8


### 9.6 Visualizing Product Segments

- **Bubble chart:** Frequency against Monetary, with bubble size = quantity sold.
- **Treemap:** the top 250 products by revenue, split by segment.
- **Lifecycle map:** first sale against last sale for every product, showing which products are new and which have stopped selling.

In [107]:
# 1) Bubble chart
fig = px.scatter(
    product_rfm, x="Frequency", y="Monetary", size="Units_Sold", color="Segment",
    hover_name="Product_Label",
    hover_data={"Recency": True, "Units_Sold": ":,", "Monetary": ":,.0f", "Frequency": True},
    category_orders={"Segment": p_present}, color_discrete_map=PRODUCT_SEGMENT_COLORS,
    log_x=True, log_y=True, size_max=32, opacity=0.65,
    title="Products: Frequency vs Monetary (bubble = units sold)",
)
fig.update_layout(title_x=0.5, height=560)
fig.show()

# 2) Treemap of the top 250 products by revenue
top_products = product_rfm.nlargest(250, "Monetary")
fig = px.treemap(top_products, path=[px.Constant("Top 250 products by revenue"), "Segment", "Product_Label"],
                 values="Monetary", color="Segment",
                 color_discrete_map={**PRODUCT_SEGMENT_COLORS, "(?)": "#FFFFFF"},
                 title="Top 250 products by revenue, grouped by segment")
fig.update_traces(root_color="white", textinfo="label+value")
fig.update_layout(title_x=0.5, height=620)
fig.show()

# 3) Lifecycle map: when did each product first and last sell?
fig = px.scatter(
    product_rfm, x="First_Sale", y="Last_Sale", color="Segment",
    hover_name="Product_Label", hover_data={"Frequency": True, "Monetary": ":,.0f"},
    category_orders={"Segment": p_present}, color_discrete_map=PRODUCT_SEGMENT_COLORS,
    opacity=0.6, title="Product lifecycle: first sale vs last sale",
)
fig.update_layout(title_x=0.5, height=520, xaxis_title="First sale", yaxis_title="Last sale")
fig.show()

# 4) Top products inside the two segments that need decisions
cols = ["StockCode", "Description", "Frequency", "Monetary", "Units_Sold", "Recency"]
print("Star Products (top 10 by revenue)")
display(product_rfm[product_rfm["Segment"] == "Star Products"].nlargest(10, "Monetary")[cols])
print("Dead Stock: highest-revenue products that stopped selling (review first)")
display(product_rfm[product_rfm["Segment"] == "Dead Stock"].nlargest(10, "Monetary")[cols])

Star Products (top 10 by revenue)


,StockCode,Description,Frequency,Monetary,Units_Sold,Recency
1310,22423,REGENCY CAKESTAND 3 TIER,1989,174765.13,13890,1
3407,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2203,127690.60,41664,1
2670,47566,PARTY BUNTING,1686,99524.13,18299,1
3387,85099B,JUMBO BAG RED RETROSPOT,2092,94347.98,48478,1
2020,23166,MEDIUM CERAMIC TOP STORAGE JAR,247,81700.92,78033,1
1942,23084,RABBIT NIGHT LIGHT,1007,69990.11,31722,1
1006,22086,PAPER CHAIN KIT 50'S CHRISTMAS,1160,64952.29,19355,1
3194,84879,ASSORTED COLOUR BIRD ORNAMENT,1455,59094.93,36461,1
2845,79321,CHILLI LIGHTS,667,55656.04,10423,1
1109,22197,POPCORN HOLDER,1392,51354.02,56921,1


Dead Stock: highest-revenue products that stopped selling (review first)


,StockCode,Description,Frequency,Monetary,Units_Sold,Recency
2641,47021G,SET/6 BEAD COASTERS GAUZE BAG GOLD,28,421.62,870,170
974,22051,BLUE SCANDINAVIAN PAISLEY WRAP,39,420.00,1000,15
384,21129,SILVER FISHING GNOME,27,419.52,52,11
234,20830,SILVER PHOTO FRAME,37,409.83,136,68
2582,37476,CONDIMENT TRAY 4 BOWLS AND 4 SPOONS,26,399.02,37,50
3072,84663A,GRASS HOPPER WOODEN WALL CLOCK,36,398.61,303,18
3262,84974,S/2 ZINC HEART DESIGN PLANTERS,28,398.00,40,39
112,18007,ESSENTIAL BALM 3.5g TIN IN ENVELOPE,30,395.61,5884,142
3518,85232A,SET/3 POLKADOT STACKING TINS,29,389.55,78,186
425,21197,MULTICOLOUR CONFETTI IN TUBE,27,382.96,284,8


### 9.7 Linking Customers and Products

This step joins the two analyses: **what does each main customer segment (Champions, Loyal, At Risk, Lost) buy, in terms of product segments?** and **which products do the Champions love most?** This is what turns RFM from a set of numbers into a decision — what to promote to which segment.

- The stacked bar shows, for each customer segment, the share of its revenue that comes from each product segment (every bar is scaled to 100%, so segments of different sizes can be compared).
- The horizontal bar chart lists the 10 products that bring in the most revenue from Champions.

In [108]:
cross = (
    customer_transactions[["CustomerID", "StockCode", "TotalLineRevenue"]]
    .merge(customer_rfm[["CustomerID", "Segment"]].rename(columns={"Segment": "Customer_Segment"}),
           on="CustomerID", how="left")
    .merge(product_rfm[["StockCode", "Segment"]].rename(columns={"Segment": "Product_Segment"}),
           on="StockCode", how="left")
)

cross_rev = (cross.groupby(["Customer_Segment", "Product_Segment"])["TotalLineRevenue"]
             .sum().reset_index())

fig = px.bar(cross_rev, x="Customer_Segment", y="TotalLineRevenue", color="Product_Segment",
             category_orders={"Customer_Segment": present, "Product_Segment": p_present},
             color_discrete_map=PRODUCT_SEGMENT_COLORS,
             title="What each customer segment buys (share of revenue by product segment)")
fig.update_layout(barnorm="percent", title_x=0.5, height=500,
                  yaxis_title="% of segment revenue", xaxis_title="", legend_title="Product segment")
fig.show()

# Favorite products of the best customers
champ_products = (
    cross[cross["Customer_Segment"] == "Champions"]
    .groupby("StockCode")["TotalLineRevenue"].sum()
    .nlargest(10).reset_index()
    .merge(product_rfm[["StockCode", "Product_Label", "Segment"]], on="StockCode", how="left")
    .sort_values("TotalLineRevenue")
)
if len(champ_products):
    fig = px.bar(champ_products, x="TotalLineRevenue", y="Product_Label", orientation="h",
                 color="Segment", color_discrete_map=PRODUCT_SEGMENT_COLORS,
                 title="Top 10 products bought by Champions (by revenue)",
                 labels={"TotalLineRevenue": "Revenue from Champions", "Product_Label": ""})
    fig.update_layout(title_x=0.5, height=470)
    fig.show()

In [109]:
# Save the results (they will feed the dashboard / report later)
customer_rfm.to_csv("customer_rfm_segments.csv", index=False)
product_rfm.to_csv("product_rfm_segments.csv", index=False)
print("Saved: customer_rfm_segments.csv", customer_rfm.shape, "| product_rfm_segments.csv", product_rfm.shape)

Saved: customer_rfm_segments.csv (4335, 11) | product_rfm_segments.csv (3912, 17)


## 10. Market Basket Analysis (Apriori)

RFM tells us **who** buys and **which** products perform. Apriori answers a different question: **which products are bought together in the same invoice?** This is the base for "customers who bought X also bought Y", bundles and shelf/website placement.

**Three numbers describe every rule `A → B`:**

| Metric | Meaning | Question it answers |
|---|---|---|
| **Support** | share of all baskets that contain A and B together | How common is this combination? |
| **Confidence** | of the baskets with A, the share that also contain B | If someone buys A, how likely is B? |
| **Lift** | confidence ÷ (share of baskets containing B) | Is it more than coincidence? `lift = 1` means independent, `lift > 1` means they attract each other |

**Choices I made (and why):**
- The **basket is the invoice** (`InvoiceNo`), and I use the same valid sales lines as section 9, Guests included (a basket does not need to know the buyer).
- **Quantity is ignored**: Apriori only needs to know whether a product was in the basket or not.
- **Single-product invoices are removed**, because they cannot produce a rule. Support is calculated on the remaining baskets.
- Products that cannot reach `MIN_SUPPORT` are dropped before running. This only saves memory, the result is identical.
- `MAX_LEN = 3` keeps the rules readable (at most 2 products on the left side).

In [110]:
# %pip install mlxtend
from mlxtend.frequent_patterns import apriori, association_rules

MIN_SUPPORT    = 0.02   # a combination must appear in at least 2% of the baskets
MIN_CONFIDENCE = 0.40
MIN_LIFT       = 1.5
MAX_LEN        = 3      # itemsets of up to 3 products

label_by_code = product_rfm.set_index("StockCode")["Product_Label"]


def build_basket_matrix(transactions, min_support):
    """One row per invoice, one True/False column per product."""
    items = transactions[["InvoiceNo", "StockCode"]].drop_duplicates()

    # baskets with a single product cannot produce a rule
    size = items.groupby("InvoiceNo")["StockCode"].transform("size")
    items = items[size >= 2]
    all_baskets = items["InvoiceNo"].unique()

    # drop products that can never reach min_support (saves memory, same result)
    counts = items["StockCode"].value_counts()
    keep = counts[counts >= min_support * len(all_baskets)].index
    items = items[items["StockCode"].isin(keep)]

    matrix = pd.crosstab(items["InvoiceNo"], items["StockCode"]).astype(bool)
    # baskets that lost all their items must still be counted in the denominator of support
    matrix = matrix.reindex(all_baskets, fill_value=False)
    matrix.columns = [label_by_code.get(c, str(c)) for c in matrix.columns]
    return matrix


def mine_rules(transactions, min_support, min_confidence=MIN_CONFIDENCE, min_lift=MIN_LIFT):
    """Frequent itemsets -> association rules (filtered by lift, sorted by lift)."""
    basket = build_basket_matrix(transactions, min_support)
    freq = apriori(basket, min_support=min_support, use_colnames=True, max_len=MAX_LEN,
                  low_memory=True)   # candidates are checked one by one instead of in one huge array
    if freq.empty:
        return basket, freq, pd.DataFrame()

    try:     # newer mlxtend versions need the number of itemsets
        rules = association_rules(freq, num_itemsets=len(basket), metric="confidence", min_threshold=min_confidence)
    except TypeError:
        rules = association_rules(freq, metric="confidence", min_threshold=min_confidence)

    if rules.empty:
        return basket, freq, pd.DataFrame()
    rules = rules[rules["lift"] >= min_lift].copy()
    rules["Antecedent"] = rules["antecedents"].apply(lambda s: " + ".join(sorted(s)))
    rules["Consequent"] = rules["consequents"].apply(lambda s: " + ".join(sorted(s)))
    rules["Rule"] = rules["Antecedent"] + "  →  " + rules["Consequent"]
    return basket, freq, rules.sort_values("lift", ascending=False).reset_index(drop=True)

### 10.1 Running Apriori on All Baskets

In [111]:
basket_matrix, freq_itemsets, rules = mine_rules(product_transactions, MIN_SUPPORT)

print(f"Baskets with 2+ products          : {len(basket_matrix):,}")
print(f"Products that reach {MIN_SUPPORT:.0%} support    : {basket_matrix.shape[1]:,}")
print(f"Frequent itemsets                 : {len(freq_itemsets):,}")
print(f"Rules (confidence >= {MIN_CONFIDENCE:.0%}, lift >= {MIN_LIFT}) : {len(rules):,}")

rule_cols = ["Antecedent", "Consequent", "support", "confidence", "lift"]
if len(rules):
    display(rules[rule_cols].head(15).round(3))
else:
    print("No rules at these thresholds - lower MIN_SUPPORT / MIN_CONFIDENCE / MIN_LIFT.")

Baskets with 2+ products          : 18,292
Products that reach 2% support    : 343
Frequent itemsets                 : 470
Rules (confidence >= 40%, lift >= 1.5) : 162


,Antecedent,Consequent,support,confidence,lift
0,22578 - WOODEN STAR CHRISTMAS SCANDINAVIAN,22577 - WOODEN HEART CHRISTMAS SCANDINAVIAN,0.021,0.759,25.816
1,22577 - WOODEN HEART CHRISTMAS SCANDINAVIAN,22578 - WOODEN STAR CHRISTMAS SCANDINAVIAN,0.021,0.721,25.816
2,22698 - PINK REGENCY TEACUP AND SAUCER,22697 - GREEN REGENCY TEACUP AND SAUCER + 22699 - ROSES REGENCY TEACUP AND SAUCER,0.030,0.712,16.963
3,22697 - GREEN REGENCY TEACUP AND SAUCER + 22699 - ROSES REGENCY TEACUP AND SAUCER,22698 - PINK REGENCY TEACUP AND SAUCER,0.030,0.706,16.963
4,22697 - GREEN REGENCY TEACUP AND SAUCER,22698 - PINK REGENCY TEACUP AND SAUCER + 22699 - ROSES REGENCY TEACUP AND SAUCER,0.030,0.534,16.307
5,22698 - PINK REGENCY TEACUP AND SAUCER + 22699 - ROSES REGENCY TEACUP AND SAUCER,22697 - GREEN REGENCY TEACUP AND SAUCER,0.030,0.905,16.307
6,22697 - GREEN REGENCY TEACUP AND SAUCER,22698 - PINK REGENCY TEACUP AND SAUCER,0.035,0.624,14.990
7,22698 - PINK REGENCY TEACUP AND SAUCER,22697 - GREEN REGENCY TEACUP AND SAUCER,0.035,0.832,14.990
8,22697 - GREEN REGENCY TEACUP AND SAUCER + 22698 - PINK REGENCY TEACUP AND SAUCER,22699 - ROSES REGENCY TEACUP AND SAUCER,0.030,0.856,14.734
9,22699 - ROSES REGENCY TEACUP AND SAUCER,22697 - GREEN REGENCY TEACUP AND SAUCER + 22698 - PINK REGENCY TEACUP AND SAUCER,0.030,0.510,14.734


### 10.2 Visualizing the Rules

- **Left:** every rule as a point. Rules in the top-right corner (high support and high confidence) are the ones worth acting on first. Colour and size show the lift.
- **Right:** the strongest combinations by lift. `A → B` and `B → A` have the same lift, so each pair is shown **once** (the direction with the higher confidence).

In [112]:
if len(rules):
    fig = px.scatter(rules, x="support", y="confidence", color="lift", size="lift",
                     hover_name="Rule", color_continuous_scale="Viridis", opacity=0.75,
                     title="Association rules: support vs confidence (colour = lift)")
    fig.update_layout(title_x=0.5, height=500)
    fig.show()

    # one row per unique combination of products
    unique_sets = (rules.assign(Items=rules.apply(lambda r: frozenset(r["antecedents"] | r["consequents"]), axis=1))
                        .sort_values(["lift", "confidence"], ascending=False)
                        .drop_duplicates("Items")
                        .head(15).sort_values("lift"))
    fig = px.bar(unique_sets, x="lift", y="Rule", orientation="h", color="confidence",
                 color_continuous_scale="Blues", hover_data=["support", "confidence"],
                 title="Top 15 product combinations by lift")
    fig.update_layout(title_x=0.5, height=620, yaxis_title="", yaxis=dict(automargin=True))
    fig.show()

### 10.3 Linking Apriori with RFM — What Do the Champions Buy Together?

The same analysis, but only on the invoices of the **Champions** segment from section 8. If their combinations differ from the all-customer ones, those are the bundles and recommendations worth showing to the best customers. Champions have far fewer invoices, so I use a slightly **higher support** to avoid rules built on a handful of baskets.

In [113]:
MIN_SUPPORT_CHAMPIONS = 0.03

champion_ids = customer_rfm.loc[customer_rfm["Segment"] == "Champions", "CustomerID"]
champion_tx = product_transactions[product_transactions["CustomerID"].isin(champion_ids)]
print(f"Champions invoices: {champion_tx['InvoiceNo'].nunique():,}")

champ_matrix, champ_freq, champ_rules = mine_rules(champion_tx, MIN_SUPPORT_CHAMPIONS)
print(f"Baskets with 2+ products: {len(champ_matrix):,} | rules: {len(champ_rules):,}")
if len(champ_rules):
    display(champ_rules[rule_cols].head(10).round(3))
else:
    print("No rules for Champions at these thresholds.")

Champions invoices: 7,848
Baskets with 2+ products: 7,266 | rules: 20


,Antecedent,Consequent,support,confidence,lift
0,22697 - GREEN REGENCY TEACUP AND SAUCER,22699 - ROSES REGENCY TEACUP AND SAUCER,0.032,0.789,16.110
1,22699 - ROSES REGENCY TEACUP AND SAUCER,22697 - GREEN REGENCY TEACUP AND SAUCER,0.032,0.663,16.110
2,23300 - GARDENERS KNEELING PAD CUP OF TEA,23301 - GARDENERS KNEELING PAD KEEP CALM,0.033,0.746,14.612
3,23301 - GARDENERS KNEELING PAD KEEP CALM,23300 - GARDENERS KNEELING PAD CUP OF TEA,0.033,0.642,14.612
4,22726 - ALARM CLOCK BAKELIKE GREEN,22727 - ALARM CLOCK BAKELIKE RED,0.037,0.658,11.486
5,22727 - ALARM CLOCK BAKELIKE RED,22726 - ALARM CLOCK BAKELIKE GREEN,0.037,0.642,11.486
6,22384 - LUNCH BAG PINK POLKADOT,20727 - LUNCH BAG BLACK SKULL.,0.033,0.538,7.548
7,20727 - LUNCH BAG BLACK SKULL.,22384 - LUNCH BAG PINK POLKADOT,0.033,0.463,7.548
8,20725 - LUNCH BAG RED RETROSPOT,22384 - LUNCH BAG PINK POLKADOT,0.037,0.419,6.824
9,22384 - LUNCH BAG PINK POLKADOT,20725 - LUNCH BAG RED RETROSPOT,0.037,0.608,6.824


In [114]:
# Save the rules (they can feed the dashboard / report later)
if len(rules):
    rules[rule_cols].to_csv("association_rules.csv", index=False)
    print("Saved: association_rules.csv", rules[rule_cols].shape)

Saved: association_rules.csv (162, 5)


### 10.4 How to Read the Result and Its Limits

- **Act on lift first, then check confidence and support.** A rule with a huge lift but tiny support is real but touches very few baskets; a rule with high support but lift near 1 is just two popular products.
- Rules show **association, not cause**: they say what is bought together, not that promoting A will make people buy B.
- **Wholesale invoices** contain many products, so they create many combinations and can inflate support. Wholesalers often buy a whole *family* of the same item in different colours, so many of the strongest rules are probably variants of one product. Treat them as "range" information, not as cross-sell between different products.
- The data covers one year, so seasonal gift items can look more related than they are.
- Using `StockCode` as the item keeps colour variants separate. Grouping variants by their description would give fewer but broader rules.

## 11. Summary & Limitations

**What this notebook did**

1. Cleaned the data with documented decisions: separated non-product lines, standardised descriptions, recovered zero prices, and *flagged* guests and duplicates instead of deleting them.
2. Ran an interactive EDA with Plotly.
3. Built **Customer RFM** and split customers into **4 main segments (Champions, Loyal, At Risk, Lost)** made of 11 sub-segments, connected to business actions. The charts include filters so that large segments and outliers do not hide the rest.
4. Built **Product RFM**, segmented the products, and linked both analyses.
5. Ran **Apriori** market-basket analysis to find products bought together, for all baskets and for the Champions.

**Limitations (known and accepted)**

1. **Tie-breaking:** `rank(method="first")` places customers with identical values (especially Frequency = 1) into neighbouring scores based on row order — a small bias.
2. **Monetary is gross sales**, without subtracting returns, because returns are analysed separately (section 7.9). Section 7.9.7 builds `customer_returns` with a per-customer `Net_Value` that can be used for a net-Monetary version.
3. **Guest checkouts** (24.68% of product lines) are not part of the customer RFM because they have no identity.
4. **Recovered prices** are estimates (about 0.2% of rows), not real prices.
5. **The data covers one year only**, so seasonality cannot be separated from trend, which affects Product Recency in particular.
6. **The segments are rule-based.** The 4-segment roll-up (`SEGMENT_MAP`) is a business choice; the natural next step is to compare it with a clustering approach (K-Means) and check whether a similar structure appears.
7. **Apriori works on presence/absence** of a product in an invoice (no quantity or price), and wholesale invoices can inflate support, so many rules may just be colour variants of the same product.
